# MSCAD regeneration pipeline — self-contained notebook

Regenerates flow records **with flow identifiers** (Src/Dst IP, ports, protocol,
timestamp) from the raw MSCAD PCAPs, verifies the extraction, compares it against the
original Kaggle `MSCAD.csv`, and writes the consolidated CSVs plus a report.

The Kaggle file carries the CICFlowMeter features and a `Label` but no identifiers, so
PrefixSpan sequence mining and the attack-state graph cannot be built from it. This
supplies them.

**Everything is in this notebook.** No other file is needed — no `.py` imports, no
`pip install` beyond what Anaconda already has (`numpy`, `scipy`, `pandas`).

### How to use it

1. Run section 1 (environment check).
2. Edit section 2 (paths) — the only cell you should need to change.
3. Run sections 3–5 once; they only define code.
4. Run section 6 cell by cell: inspect inputs, N-0 test, all six, verify, compare,
   consolidate, report.
5. Section 7 loads the result and builds the per-host-pair label sequences.

Memory is safe in-kernel: the engine keeps no packets, only per-flow accumulators, and
a 704,000-packet capture was measured at a flat 53 MB. If you still want a clean slate
between stages, use *Kernel → Restart* and re-run sections 2–5.

## 1. Environment check

In [ ]:
import os, sys, platform, importlib

print("Python    :", sys.version.split()[0])
print("executable:", sys.executable)
print("platform  :", platform.platform())
if sys.version_info < (3, 8):
    print("\n*** Python 3.8+ required ***")

missing = []
for pkg in ("numpy", "scipy", "pandas"):
    try:
        m = importlib.import_module(pkg)
        print("%-8s  : %s" % (pkg, getattr(m, "__version__", "?")))
    except ImportError:
        missing.append(pkg)
        print("%-8s  : MISSING" % pkg)
try:
    import scapy
    print("%-8s  : %s (optional)" % ("scapy", scapy.__version__))
except ImportError:
    print("%-8s  : not installed (optional, only for .pcapng input)" % "scapy")

if missing:
    print("\nInstall into THIS kernel with:")
    print("    %s -m pip install %s" % (sys.executable, " ".join(missing)))
else:
    print("\nAll required packages present.")

## 2. Configuration — edit this cell

Paths first. Everything below them is a documented semantic switch; the defaults match
the **Java** CICFlowMeter, which is the tool that produced the Kaggle `MSCAD.csv` we
compare against. Section 6.8 of this notebook tells you which switch to change if a
feature family looks wrong.

In [ ]:
import os


class CFG:
    # ---------------- PATHS: edit these ------------------------------
    PCAP_DIR_CANDIDATES = [
        r"D:\IDSPROJECT2026\MSCAD\MSCAD Dataset\PCAPs\pcapngtopcap",
        r"D:\IDSPROJECT2026\MSCAD\MSCAD Dataset\MSCAD Dataset\PCAPs\PCAPs\pcapngtopcap",
    ]
    PCAP_SEARCH_ROOT = r"D:\IDSPROJECT2026\MSCAD"
    CSV_ORIGINAL_DIR = r"D:\IDSPROJECT2026\MSCAD\CSV_ORIGINAL"
    OUT_DIR = r"D:\IDSPROJECT2026\MSCAD\MSCAD_REGENERATED"
    TSHARK_EXE = r"C:\Program Files\Wireshark\tshark.exe"   # optional

    RAW_SUBDIR = "raw"
    METRICS_SUBDIR = "metrics"

    # ---------------- captures --------------------------------------
    # key, official label (dataset's own spelling), description
    CAPTURES = [
        ("N-0",    "Normal",        "normal traffic"),
        ("Scan-1", "Port_Scan",     "Full / SYN / FIN / UDP port scans"),
        ("App-01", "DoS slowloris", "application-layer DDoS (HTTP Slowloris)"),
        ("App-02", "ICMP_Flood",    "volume-based DDoS (ICMP flood, fragmented pings)"),
        ("W-B-01", "Web_Crwling",   "web crawling"),
        ("W-B-02", "Brute_Force",   "password cracking"),
    ]
    BENIGN_LABEL = "Benign"
    PCAP_EXTENSIONS = [".pcap", ".pcapng", ".cap"]

    # ---------------- timeouts (seconds) ----------------------------
    FLOW_TIMEOUT = 120.0      # a flow older than this is cut, a new one starts
    IDLE_TIMEOUT = 240.0      # a gap this long also cuts the flow
    ACTIVITY_TIMEOUT = 5.0    # Active/Idle segmentation threshold
    CLUMP_TIMEOUT = 1.0       # subflow + bulk clump threshold
    BULK_BOUND = 4            # packets before a burst counts as a "bulk"
    CLOSE_FLOW_ON_FIN = True  # Java CICFlowMeter behaviour
    CLOSE_FLOW_ON_RST = False # Java does not close on RST

    SWEEP_EVERY_PACKETS = 50000
    PROGRESS_EVERY_PACKETS = 100000

    # ---------------- semantic switches -----------------------------
    # "payload" = L4 payload bytes (Java getPayloadBytes); "frame" = whole
    # captured frame (pip cicflowmeter len(packet)). The CIC CSV family has
    # ZEROS in Fwd/Bwd Pkt Len Min, which only happens under payload bytes.
    PKT_LEN_MODE = "payload"
    # "l4" = transport header only (Java); "ip" = IP header (pip ihl*4)
    HEADER_LEN_MODE = "l4"
    # 1 = bias-corrected n-1 (Java commons-math); 0 = population n (numpy)
    STD_DDOF = 1
    # "divide" = total / (1 + >1s gaps), as Java; "copy" = Subflow == total, as pip
    SUBFLOW_MODE = "divide"
    # Java truncates long/long for Down/Up Ratio, Subflow and bulk averages
    JAVA_INTEGER_DIVISION = True
    # Init Fwd/Bwd Win Byts when that direction has no TCP packet (CIC uses -1)
    INIT_WIN_DEFAULT = -1
    # Java drops 1-packet flows on timeout; we keep them so every packet is
    # accounted for in the Part 2 reconciliation
    DROP_SINGLE_PACKET_FLOWS = False
    # Java does not add the final unterminated Active segment
    ADD_FINAL_ACTIVE_SEGMENT = False

    # ---------------- labelling -------------------------------------
    ATTACKER_SHARE = 0.50     # Src IP is an attacker at >= this share of top sender
    ATTACKER_TOP_N_PRINT = 5

    # ---------------- output ----------------------------------------
    TIME_UNIT_SCALE = 1000000.0          # time features are in MICROSECONDS
    TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S.%f"
    TIMESTAMP_UTC = True

    MATCH_KEY_FEATURES = [
        "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts",
        "TotLen Bwd Pkts", "Fwd Pkt Len Max", "Bwd Pkt Len Max",
        "Flow IAT Max", "Flow IAT Min", "Fwd Header Len", "Bwd Header Len",
        "Init Bwd Win Byts",
    ]
    IDENTIFIER_COLUMNS = ["Src IP", "Dst IP", "Src Port", "Dst Port",
                          "Protocol", "Timestamp"]
    LABEL_COLUMN = "Label"

    # ---------------- helpers ---------------------------------------
    @staticmethod
    def resolve_pcap_dir(verbose=True):
        """Return (path, how). Falls back to searching for 'pcapngtopcap'."""
        for cand in CFG.PCAP_DIR_CANDIDATES:
            if os.path.isdir(cand):
                return cand, "configured candidate"
        if os.path.isdir(CFG.PCAP_SEARCH_ROOT):
            hits = []
            for root, dirs, _files in os.walk(CFG.PCAP_SEARCH_ROOT):
                for d in dirs:
                    if d.lower() == "pcapngtopcap":
                        hits.append(os.path.join(root, d))
            if hits:
                hits.sort(key=len)
                if verbose and len(hits) > 1:
                    print("  NOTE: several pcapngtopcap folders found:")
                    for h in hits:
                        print("        " + h)
                return hits[0], "recursive search under %s" % CFG.PCAP_SEARCH_ROOT
        return None, "not found"

    @staticmethod
    def ensure_out_dirs():
        for d in (CFG.OUT_DIR,
                  os.path.join(CFG.OUT_DIR, CFG.RAW_SUBDIR),
                  os.path.join(CFG.OUT_DIR, CFG.METRICS_SUBDIR)):
            if not os.path.isdir(d):
                os.makedirs(d)
        return CFG.OUT_DIR

    @staticmethod
    def raw_csv_path(capture_key):
        return os.path.join(CFG.OUT_DIR, CFG.RAW_SUBDIR,
                            "%s_flows.csv" % capture_key)

    @staticmethod
    def metrics_path(name):
        return os.path.join(CFG.OUT_DIR, CFG.METRICS_SUBDIR, name)

    @staticmethod
    def find_capture_file(pcap_dir, stem):
        """Find a capture file for a stem, case-insensitively."""
        if not pcap_dir or not os.path.isdir(pcap_dir):
            return None
        entries = os.listdir(pcap_dir)
        lower = dict((e.lower(), e) for e in entries)
        for ext in CFG.PCAP_EXTENSIONS:
            want = (stem + ext).lower()
            if want in lower:
                return os.path.join(pcap_dir, lower[want])
        for e in entries:
            base, ext = os.path.splitext(e)
            if base.lower() == stem.lower() and ext.lower() in CFG.PCAP_EXTENSIONS:
                return os.path.join(pcap_dir, e)
        return None


print("config loaded")
print("  out dir :", CFG.OUT_DIR)
for lbl, p in (("pcap dir", CFG.PCAP_DIR_CANDIDATES[0]),
               ("csv dir ", CFG.CSV_ORIGINAL_DIR)):
    print("  %s: %s   %s" % (lbl, p, "EXISTS" if os.path.isdir(p) else "*** MISSING ***"))
print("  tshark  : %s   %s" % (CFG.TSHARK_EXE,
                               "found" if os.path.isfile(CFG.TSHARK_EXE) else "not found (optional)"))
print("\ntimeouts: flow %.0fs | idle %.0fs | activity %.0fs | clump %.0fs"
      % (CFG.FLOW_TIMEOUT, CFG.IDLE_TIMEOUT, CFG.ACTIVITY_TIMEOUT, CFG.CLUMP_TIMEOUT))
print("modes   : pkt-len %r | header-len %r | std ddof %d | subflow %r"
      % (CFG.PKT_LEN_MODE, CFG.HEADER_LEN_MODE, CFG.STD_DDOF, CFG.SUBFLOW_MODE))
print("time features in MICROSECONDS")

## 3. Feature name tables

The 76 CICFlowMeter features with every alias (CICFlowMeter V3 / CIC-IDS2017 and
V4 / CSE-CIC-IDS2018 naming). The pipeline never hard-codes "the 66 MSCAD columns":
it reads the real header from your Kaggle file and resolves each name through
`NAME_TO_KEY`, so whichever 66 of the 76 MSCAD carries, in whatever order, the output
matches. Any header name that cannot be resolved is reported, never guessed.

In [ ]:
import re

# --------------------------------------------------------------------------
# The 76 CICFlowMeter features, in canonical CSE-CIC-IDS2018 column order.
# (display name, internal key)
# --------------------------------------------------------------------------
CANONICAL_FEATURES = [
    ("Flow Duration",        "flow_duration"),
    ("Tot Fwd Pkts",         "tot_fwd_pkts"),
    ("Tot Bwd Pkts",         "tot_bwd_pkts"),
    ("TotLen Fwd Pkts",      "totlen_fwd_pkts"),
    ("TotLen Bwd Pkts",      "totlen_bwd_pkts"),
    ("Fwd Pkt Len Max",      "fwd_pkt_len_max"),
    ("Fwd Pkt Len Min",      "fwd_pkt_len_min"),
    ("Fwd Pkt Len Mean",     "fwd_pkt_len_mean"),
    ("Fwd Pkt Len Std",      "fwd_pkt_len_std"),
    ("Bwd Pkt Len Max",      "bwd_pkt_len_max"),
    ("Bwd Pkt Len Min",      "bwd_pkt_len_min"),
    ("Bwd Pkt Len Mean",     "bwd_pkt_len_mean"),
    ("Bwd Pkt Len Std",      "bwd_pkt_len_std"),
    ("Flow Byts/s",          "flow_byts_s"),
    ("Flow Pkts/s",          "flow_pkts_s"),
    ("Flow IAT Mean",        "flow_iat_mean"),
    ("Flow IAT Std",         "flow_iat_std"),
    ("Flow IAT Max",         "flow_iat_max"),
    ("Flow IAT Min",         "flow_iat_min"),
    ("Fwd IAT Tot",          "fwd_iat_tot"),
    ("Fwd IAT Mean",         "fwd_iat_mean"),
    ("Fwd IAT Std",          "fwd_iat_std"),
    ("Fwd IAT Max",          "fwd_iat_max"),
    ("Fwd IAT Min",          "fwd_iat_min"),
    ("Bwd IAT Tot",          "bwd_iat_tot"),
    ("Bwd IAT Mean",         "bwd_iat_mean"),
    ("Bwd IAT Std",          "bwd_iat_std"),
    ("Bwd IAT Max",          "bwd_iat_max"),
    ("Bwd IAT Min",          "bwd_iat_min"),
    ("Fwd PSH Flags",        "fwd_psh_flags"),
    ("Bwd PSH Flags",        "bwd_psh_flags"),
    ("Fwd URG Flags",        "fwd_urg_flags"),
    ("Bwd URG Flags",        "bwd_urg_flags"),
    ("Fwd Header Len",       "fwd_header_len"),
    ("Bwd Header Len",       "bwd_header_len"),
    ("Fwd Pkts/s",           "fwd_pkts_s"),
    ("Bwd Pkts/s",           "bwd_pkts_s"),
    ("Pkt Len Min",          "pkt_len_min"),
    ("Pkt Len Max",          "pkt_len_max"),
    ("Pkt Len Mean",         "pkt_len_mean"),
    ("Pkt Len Std",          "pkt_len_std"),
    ("Pkt Len Var",          "pkt_len_var"),
    ("FIN Flag Cnt",         "fin_flag_cnt"),
    ("SYN Flag Cnt",         "syn_flag_cnt"),
    ("RST Flag Cnt",         "rst_flag_cnt"),
    ("PSH Flag Cnt",         "psh_flag_cnt"),
    ("ACK Flag Cnt",         "ack_flag_cnt"),
    ("URG Flag Cnt",         "urg_flag_cnt"),
    ("CWE Flag Count",       "cwr_flag_count"),
    ("ECE Flag Cnt",         "ece_flag_cnt"),
    ("Down/Up Ratio",        "down_up_ratio"),
    ("Pkt Size Avg",         "pkt_size_avg"),
    ("Fwd Seg Size Avg",     "fwd_seg_size_avg"),
    ("Bwd Seg Size Avg",     "bwd_seg_size_avg"),
    ("Fwd Byts/b Avg",       "fwd_byts_b_avg"),
    ("Fwd Pkts/b Avg",       "fwd_pkts_b_avg"),
    ("Fwd Blk Rate Avg",     "fwd_blk_rate_avg"),
    ("Bwd Byts/b Avg",       "bwd_byts_b_avg"),
    ("Bwd Pkts/b Avg",       "bwd_pkts_b_avg"),
    ("Bwd Blk Rate Avg",     "bwd_blk_rate_avg"),
    ("Subflow Fwd Pkts",     "subflow_fwd_pkts"),
    ("Subflow Fwd Byts",     "subflow_fwd_byts"),
    ("Subflow Bwd Pkts",     "subflow_bwd_pkts"),
    ("Subflow Bwd Byts",     "subflow_bwd_byts"),
    ("Init Fwd Win Byts",    "init_fwd_win_byts"),
    ("Init Bwd Win Byts",    "init_bwd_win_byts"),
    ("Fwd Act Data Pkts",    "fwd_act_data_pkts"),
    ("Fwd Seg Size Min",     "fwd_seg_size_min"),
    ("Active Mean",          "active_mean"),
    ("Active Std",           "active_std"),
    ("Active Max",           "active_max"),
    ("Active Min",           "active_min"),
    ("Idle Mean",            "idle_mean"),
    ("Idle Std",             "idle_std"),
    ("Idle Max",             "idle_max"),
    ("Idle Min",             "idle_min"),
]

CANONICAL_DISPLAY = [d for d, _k in CANONICAL_FEATURES]
CANONICAL_KEYS = [k for _d, k in CANONICAL_FEATURES]

# --------------------------------------------------------------------------
# Alternate spellings (CICFlowMeter V3 / CIC-IDS2017 and common variants).
# --------------------------------------------------------------------------
ALIASES = {
    "flow_duration":      ["Flow Duration"],
    "tot_fwd_pkts":       ["Total Fwd Packets", "Total Fwd Pkts", "Tot Fwd Pkt"],
    "tot_bwd_pkts":       ["Total Backward Packets", "Total Bwd Packets", "Tot Bwd Pkt"],
    "totlen_fwd_pkts":    ["Total Length of Fwd Packets", "TotLen Fwd Pkt",
                           "Total Length of Fwd Packet"],
    "totlen_bwd_pkts":    ["Total Length of Bwd Packets", "TotLen Bwd Pkt",
                           "Total Length of Bwd Packet"],
    "fwd_pkt_len_max":    ["Fwd Packet Length Max"],
    "fwd_pkt_len_min":    ["Fwd Packet Length Min"],
    "fwd_pkt_len_mean":   ["Fwd Packet Length Mean"],
    "fwd_pkt_len_std":    ["Fwd Packet Length Std"],
    "bwd_pkt_len_max":    ["Bwd Packet Length Max"],
    "bwd_pkt_len_min":    ["Bwd Packet Length Min"],
    "bwd_pkt_len_mean":   ["Bwd Packet Length Mean"],
    "bwd_pkt_len_std":    ["Bwd Packet Length Std"],
    "flow_byts_s":        ["Flow Bytes/s", "Flow Byts/sec"],
    "flow_pkts_s":        ["Flow Packets/s", "Flow Pkts/sec"],
    "flow_iat_mean":      ["Flow IAT Mean"],
    "flow_iat_std":       ["Flow IAT Std"],
    "flow_iat_max":       ["Flow IAT Max"],
    "flow_iat_min":       ["Flow IAT Min"],
    "fwd_iat_tot":        ["Fwd IAT Total"],
    "bwd_iat_tot":        ["Bwd IAT Total"],
    "fwd_header_len":     ["Fwd Header Length"],
    "bwd_header_len":     ["Bwd Header Length"],
    "fwd_pkts_s":         ["Fwd Packets/s"],
    "bwd_pkts_s":         ["Bwd Packets/s"],
    "pkt_len_min":        ["Min Packet Length", "Packet Length Min"],
    "pkt_len_max":        ["Max Packet Length", "Packet Length Max"],
    "pkt_len_mean":       ["Packet Length Mean"],
    "pkt_len_std":        ["Packet Length Std"],
    "pkt_len_var":        ["Packet Length Variance", "Pkt Len Variance"],
    "fin_flag_cnt":       ["FIN Flag Count"],
    "syn_flag_cnt":       ["SYN Flag Count"],
    "rst_flag_cnt":       ["RST Flag Count"],
    "psh_flag_cnt":       ["PSH Flag Count"],
    "ack_flag_cnt":       ["ACK Flag Count"],
    "urg_flag_cnt":       ["URG Flag Count"],
    "cwr_flag_count":     ["CWE Flag Cnt", "CWR Flag Count", "CWR Flag Cnt",
                           "CWE Flag Count Count"],
    "ece_flag_cnt":       ["ECE Flag Count"],
    "pkt_size_avg":       ["Average Packet Size", "Avg Packet Size"],
    "fwd_seg_size_avg":   ["Avg Fwd Segment Size", "Fwd Segment Size Avg"],
    "bwd_seg_size_avg":   ["Avg Bwd Segment Size", "Bwd Segment Size Avg"],
    "fwd_byts_b_avg":     ["Fwd Avg Bytes/Bulk", "Fwd Bytes/Bulk Avg"],
    "fwd_pkts_b_avg":     ["Fwd Avg Packets/Bulk", "Fwd Packets/Bulk Avg"],
    "fwd_blk_rate_avg":   ["Fwd Avg Bulk Rate", "Fwd Bulk Rate Avg"],
    "bwd_byts_b_avg":     ["Bwd Avg Bytes/Bulk", "Bwd Bytes/Bulk Avg"],
    "bwd_pkts_b_avg":     ["Bwd Avg Packets/Bulk", "Bwd Packets/Bulk Avg"],
    "bwd_blk_rate_avg":   ["Bwd Avg Bulk Rate", "Bwd Bulk Rate Avg"],
    "subflow_fwd_pkts":   ["Subflow Fwd Packets"],
    "subflow_fwd_byts":   ["Subflow Fwd Bytes"],
    "subflow_bwd_pkts":   ["Subflow Bwd Packets"],
    "subflow_bwd_byts":   ["Subflow Bwd Bytes"],
    "init_fwd_win_byts":  ["Init_Win_bytes_forward", "Init Win bytes forward",
                           "Init Fwd Win Bytes"],
    "init_bwd_win_byts":  ["Init_Win_bytes_backward", "Init Win bytes backward",
                           "Init Bwd Win Bytes"],
    "fwd_act_data_pkts":  ["act_data_pkt_fwd", "Fwd Act Data Packets",
                           "Act Data Pkt Fwd"],
    "fwd_seg_size_min":   ["min_seg_size_forward", "Min Seg Size Forward",
                           "Fwd Seg Size Minimum"],
    "active_mean":        ["Active Mean"],
    "active_std":         ["Active Std"],
    "active_max":         ["Active Max"],
    "active_min":         ["Active Min"],
    "idle_mean":          ["Idle Mean"],
    "idle_std":           ["Idle Std"],
    "idle_max":           ["Idle Max"],
    "idle_min":           ["Idle Min"],
}

# Identifier / label columns that are not features.
IDENTIFIER_ALIASES = {
    "src_ip":    ["Src IP", "Source IP", "Src_IP"],
    "dst_ip":    ["Dst IP", "Destination IP", "Dst_IP"],
    "src_port":  ["Src Port", "Source Port", "Src_Port"],
    "dst_port":  ["Dst Port", "Destination Port", "Dst_Port"],
    "protocol":  ["Protocol"],
    "timestamp": ["Timestamp"],
    "flow_id":   ["Flow ID", "FlowID"],
    "label":     ["Label", "Labels", " Label"],
}


def normalise(name):
    """Canonical form for header matching: lowercase, alphanumerics only.

    Strips surrounding whitespace and quotes first, so ' "Flow Byts/s" '
    and 'Flow Byts/s' and 'flow_byts_s' all collapse together.
    """
    s = str(name).strip().strip('"').strip("'").strip()
    return re.sub(r"[^a-z0-9]", "", s.lower())


def _build_lookup():
    table = {}
    for display, key in CANONICAL_FEATURES:
        table[normalise(display)] = key
        table[normalise(key)] = key
    for key, alts in ALIASES.items():
        for alt in alts:
            table[normalise(alt)] = key
    for key, alts in IDENTIFIER_ALIASES.items():
        for alt in alts:
            table[normalise(alt)] = key
        table[normalise(key)] = key
    return table


NAME_TO_KEY = _build_lookup()

NON_FEATURE_KEYS = set(IDENTIFIER_ALIASES.keys())


def resolve_header(header):
    """Split an original CSV header into feature / identifier / unknown.

    Returns (feature_pairs, identifier_pairs, unknown_names) where
    feature_pairs is [(original_name, internal_key), ...] in the order the
    original file has them.
    """
    features, identifiers, unknown = [], [], []
    for raw in header:
        key = NAME_TO_KEY.get(normalise(raw))
        if key is None:
            unknown.append(raw)
        elif key in NON_FEATURE_KEYS:
            identifiers.append((raw, key))
        else:
            features.append((raw, key))
    return features, identifiers, unknown


# Features whose CICFlowMeter definition is integer-valued.  Used for the
# Part 3 row-matching key and for compact CSV formatting.
INTEGER_KEYS = {
    "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts", "totlen_fwd_pkts",
    "totlen_bwd_pkts", "fwd_pkt_len_max", "fwd_pkt_len_min",
    "bwd_pkt_len_max", "bwd_pkt_len_min", "pkt_len_max", "pkt_len_min",
    "fwd_header_len", "bwd_header_len", "fwd_seg_size_min",
    "fwd_act_data_pkts", "flow_iat_max", "flow_iat_min", "fwd_iat_tot",
    "fwd_iat_max", "fwd_iat_min", "bwd_iat_tot", "bwd_iat_max",
    "bwd_iat_min", "fwd_psh_flags", "bwd_psh_flags", "fwd_urg_flags",
    "bwd_urg_flags", "fin_flag_cnt", "syn_flag_cnt", "rst_flag_cnt",
    "psh_flag_cnt", "ack_flag_cnt", "urg_flag_cnt", "ece_flag_cnt",
    "cwr_flag_count", "init_fwd_win_byts", "init_bwd_win_byts",
    "active_max", "active_min", "idle_max", "idle_min",
    "subflow_fwd_pkts", "subflow_bwd_pkts", "subflow_fwd_byts",
    "subflow_bwd_byts", "down_up_ratio",
}

## 4. The patched flow engine

Feature definitions come from the Python `cicflowmeter` 0.5.0 package, reimplemented
here as a streaming engine keeping that package's own field names, with numeric
definitions following the Java CICFlowMeter.

A reimplementation rather than a monkey-patch was necessary: the package's feature
classes take live scapy `Packet` objects and keep every packet of a flow in
`Flow.packets`, so including non-TCP/UDP traffic, dropping the retained packets and
reading real TCP flags all change its *data model*, not just its behaviour.

**Patches 1–8** (as specified) plus **two more bugs found by reading the 0.5.0 source**:
its `update_subflow` compares `packet.time` against an already-advanced
`latest_timestamp`, so every Active/Idle feature came out identically 0; and
`get_statistics` returns zeros unless `len > 1`, so a two-packet flow reported
Flow IAT Mean/Max/Min = 0. Both are fixed. Full notes are in the cell's docstrings.

In [ ]:
import math
import os
import socket
import struct


# TCP flag bits
F_FIN, F_SYN, F_RST, F_PSH, F_ACK, F_URG, F_ECE, F_CWR = (
    0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80)

FORWARD, REVERSE = 0, 1

US = 1_000_000  # microseconds per second


# ==========================================================================
# Incremental statistics
# ==========================================================================
class RunningStats:
    """Welford accumulator with min/max/sum. No sample list is retained."""

    __slots__ = ("n", "_mean", "_m2", "mn", "mx", "total")

    def __init__(self):
        self.n = 0
        self._mean = 0.0
        self._m2 = 0.0
        self.mn = None
        self.mx = None
        self.total = 0

    def add(self, x):
        self.n += 1
        self.total += x
        d = x - self._mean
        self._mean += d / self.n
        self._m2 += d * (x - self._mean)
        if self.mn is None or x < self.mn:
            self.mn = x
        if self.mx is None or x > self.mx:
            self.mx = x

    @property
    def mean(self):
        return self._mean if self.n else 0.0

    @property
    def var(self):
        """Variance. ddof from config: 1 = Java/commons-math, 0 = numpy."""
        if self.n == 0:
            return 0.0
        if CFG.STD_DDOF == 1:
            if self.n < 2:
                return 0.0
            return self._m2 / (self.n - 1)
        return self._m2 / self.n

    @property
    def std(self):
        v = self.var
        return math.sqrt(v) if v > 0 else 0.0

    @property
    def minimum(self):
        return self.mn if self.n else 0

    @property
    def maximum(self):
        return self.mx if self.n else 0


def _div(a, b):
    """Java-style division for the long/long features."""
    if b == 0:
        return 0
    if CFG.JAVA_INTEGER_DIVISION:
        return int(a) // int(b)
    return a / b


# ==========================================================================
# Packet reading
# ==========================================================================
class PcapFormatError(Exception):
    pass


# datalink type -> (fixed header length, ethertype offset or None)
_LINKTYPES = {
    1:   ("ether", 14),
    0:   ("null", 4),
    101: ("raw", 0),
    228: ("raw", 0),
    12:  ("raw", 0),
    113: ("sll", 16),
    276: ("sll2", 20),
}


def _l2_ipv4_offset(linkname, data, hdr_len):
    """Return the offset of the IPv4 header, or -1 if this is not IPv4."""
    if linkname == "raw":
        if len(data) >= 1 and (data[0] >> 4) == 4:
            return 0
        return -1
    if linkname == "ether":
        if len(data) < 14:
            return -1
        off = 12
        et = (data[12] << 8) | data[13]
        base = 14
        # Up to two stacked VLAN / QinQ tags.
        for _ in range(2):
            if et in (0x8100, 0x88A8, 0x9100):
                if len(data) < base + 4:
                    return -1
                et = (data[base + 2] << 8) | data[base + 3]
                base += 4
            else:
                break
        if et == 0x0800:
            return base
        return -1
    if linkname == "null":
        if len(data) < 4:
            return -1
        # BSD loopback: 32-bit address family, host byte order either way.
        fam_le = struct.unpack("<I", data[0:4])[0]
        fam_be = struct.unpack(">I", data[0:4])[0]
        if 2 in (fam_le, fam_be):
            return 4
        return -1
    if linkname == "sll":
        if len(data) < 16:
            return -1
        et = (data[14] << 8) | data[15]
        return 16 if et == 0x0800 else -1
    if linkname == "sll2":
        if len(data) < 20:
            return -1
        et = (data[0] << 8) | data[1]
        return 20 if et == 0x0800 else -1
    return -1


def iter_pcap_records(path, block=4 << 20):
    """Stream (t_us, caplen, origlen, data) out of a classic libpcap file.

    Hand-rolled on purpose: it is fast, it needs no scapy dissection, and it
    hands us the captured bytes directly (so packet lengths come from the
    capture, never from a rebuilt packet).  Yields the datalink name first.
    """
    with open(path, "rb") as fh:
        gh = fh.read(24)
        if len(gh) < 24:
            raise PcapFormatError("%s: file shorter than a pcap header" % path)
        magic = gh[:4]
        if magic == b"\xd4\xc3\xb2\xa1":
            endian, tsdiv = "<", 1          # usec, little endian
        elif magic == b"\xa1\xb2\xc3\xd4":
            endian, tsdiv = ">", 1
        elif magic == b"\x4d\x3c\xb2\xa1":
            endian, tsdiv = "<", 1000       # nsec resolution
        elif magic == b"\xa1\xb2\x3c\x4d":
            endian, tsdiv = ">", 1000
        elif magic[:4] == b"\x0a\x0d\x0d\x0a":
            raise PcapFormatError(
                "%s is a pcapng file; use the scapy fallback reader" % path)
        else:
            raise PcapFormatError(
                "%s: unrecognised pcap magic %r" % (path, magic))

        linktype = struct.unpack(endian + "I", gh[20:24])[0]
        if linktype not in _LINKTYPES:
            raise PcapFormatError(
                "%s: unsupported datalink type %d" % (path, linktype))
        linkname, _hdrlen = _LINKTYPES[linktype]
        yield linkname  # first item is the datalink name

        rec = struct.Struct(endian + "IIII")
        unpack = rec.unpack_from
        buf = bytearray()
        view_start = 0
        eof = False
        while True:
            # Top up the buffer.
            if not eof and len(buf) - view_start < (1 << 18):
                if view_start:
                    del buf[:view_start]
                    view_start = 0
                chunk = fh.read(block)
                if not chunk:
                    eof = True
                else:
                    buf += chunk
            avail = len(buf) - view_start
            if avail < 16:
                if eof:
                    break
                continue
            ts_sec, ts_frac, caplen, origlen = unpack(buf, view_start)
            if avail < 16 + caplen:
                if eof:
                    break  # truncated final record
                continue
            start = view_start + 16
            data = bytes(buf[start:start + caplen])
            view_start = start + caplen
            yield (ts_sec * US + ts_frac // tsdiv, caplen, origlen, data)


def iter_pcapng_records(path):
    """Fallback for .pcapng via scapy, same output shape as the above."""
    from scapy.utils import RawPcapNgReader

    rdr = RawPcapNgReader(path)
    linkname_sent = False
    for data, meta in rdr:
        lt = getattr(meta, "linktype", 1)
        if not linkname_sent:
            if lt not in _LINKTYPES:
                raise PcapFormatError(
                    "%s: unsupported datalink type %d" % (path, lt))
            yield _LINKTYPES[lt][0]
            linkname_sent = True
        tsresol = getattr(meta, "tsresol", 1_000_000) or 1_000_000
        raw_ts = (getattr(meta, "tshigh", 0) << 32) | getattr(meta, "tslow", 0)
        t_us = raw_ts * US // tsresol if tsresol != US else raw_ts
        wl = getattr(meta, "wirelen", len(data)) or len(data)
        yield (t_us, len(data), wl, bytes(data))
    if not linkname_sent:
        yield "ether"


def open_capture(path):
    gen = (iter_pcapng_records(path)
           if path.lower().endswith((".pcapng", ".ntar"))
           else iter_pcap_records(path))
    linkname = next(gen)
    return linkname, gen


class PacketCounters:
    __slots__ = ("total", "ipv4", "non_ipv4", "ipv4_truncated", "fragments",
                 "by_proto")

    def __init__(self):
        self.total = 0
        self.ipv4 = 0
        self.non_ipv4 = 0
        self.ipv4_truncated = 0
        self.fragments = 0
        self.by_proto = {}

    def as_dict(self):
        return {
            "packets_read": self.total,
            "ipv4_packets": self.ipv4,
            "non_ipv4_packets": self.non_ipv4,
            "ipv4_truncated_len": self.ipv4_truncated,
            "ipv4_non_first_fragments": self.fragments,
            "ipv4_by_protocol": dict(sorted(self.by_proto.items())),
        }


_inet_ntoa = socket.inet_ntoa


def parse_ipv4(data, off, caplen, counters):
    """Return a slim packet tuple, or None when this is not usable IPv4.

    (src, dst, sport, dport, proto, l4_hdr_len, payload_len, frame_len,
     tcp_flags, tcp_window, is_tcp)
    """
    avail = caplen - off
    if avail < 20:
        return None
    b0 = data[off]
    if (b0 >> 4) != 4:
        return None
    ihl = (b0 & 0x0F) * 4
    if ihl < 20 or avail < ihl:
        return None
    total_len = (data[off + 2] << 8) | data[off + 3]
    frag_word = (data[off + 6] << 8) | data[off + 7]
    frag_off = (frag_word & 0x1FFF) * 8
    proto = data[off + 9]
    src = _inet_ntoa(data[off + 12:off + 16])
    dst = _inet_ntoa(data[off + 16:off + 20])

    # Trust the IP total length when it is sane; fall back to the captured
    # bytes when the capture was snapped short or the NIC did TSO (len 0).
    if total_len == 0 or total_len > avail:
        counters.ipv4_truncated += 1
        ip_len = avail
    else:
        ip_len = total_len
    ip_payload_len = max(0, ip_len - ihl)

    counters.by_proto[proto] = counters.by_proto.get(proto, 0) + 1

    if frag_off > 0:
        # Non-first fragment: no transport header in this packet.
        counters.fragments += 1
        return (src, dst, 0, 0, proto, 0, ip_payload_len, caplen, 0, 0, False)

    l4 = off + ihl
    if proto == 6 and (caplen - l4) >= 20:
        sport = (data[l4] << 8) | data[l4 + 1]
        dport = (data[l4 + 2] << 8) | data[l4 + 3]
        dataofs = (data[l4 + 12] >> 4) * 4
        if dataofs < 20:
            dataofs = 20
        flags = data[l4 + 13]
        window = (data[l4 + 14] << 8) | data[l4 + 15]
        payload = max(0, ip_payload_len - dataofs)
        return (src, dst, sport, dport, 6, dataofs, payload, caplen,
                flags, window, True)
    if proto == 17 and (caplen - l4) >= 8:
        sport = (data[l4] << 8) | data[l4 + 1]
        dport = (data[l4 + 2] << 8) | data[l4 + 3]
        udp_len = (data[l4 + 4] << 8) | data[l4 + 5]
        payload = (max(0, udp_len - 8) if 8 <= udp_len <= ip_payload_len
                   else max(0, ip_payload_len - 8))
        return (src, dst, sport, dport, 17, 8, payload, caplen, 0, 0, False)
    if proto == 1 and (caplen - l4) >= 8:
        # ICMP: 8-byte header, no ports.  Patch 2 keeps these.
        return (src, dst, 0, 0, 1, 8, max(0, ip_payload_len - 8), caplen,
                0, 0, False)
    # Any other IP protocol, or a transport header too short to read.
    return (src, dst, 0, 0, proto, 0, ip_payload_len, caplen, 0, 0, False)


# ==========================================================================
# Flow
# ==========================================================================
class Flow:
    """One CICFlowMeter flow, maintained incrementally (patch 6)."""

    __slots__ = (
        "src_ip", "dst_ip", "src_port", "dst_port", "protocol",
        "start_t", "last_t", "last_fwd_t", "last_bwd_t",
        "fwd_len", "bwd_len", "all_len",
        "flow_iat", "fwd_iat", "bwd_iat",
        "fwd_hdr_bytes", "bwd_hdr_bytes", "min_seg_fwd",
        "fwd_act_data_pkts",
        "fin", "syn", "rst", "psh", "ack", "urg", "ece", "cwr",
        "fwd_psh", "bwd_psh", "fwd_urg", "bwd_urg",
        "init_fwd_win", "init_bwd_win",
        "start_active", "end_active", "active", "idle",
        "sf_last_t", "sf_count",
        "fb_last_t", "fb_start_tmp", "fb_count", "fb_count_tmp",
        "fb_duration", "fb_pkt_count", "fb_size", "fb_size_tmp",
        "bb_last_t", "bb_start_tmp", "bb_count", "bb_count_tmp",
        "bb_duration", "bb_pkt_count", "bb_size", "bb_size_tmp",
    )

    def __init__(self, pkt, t_us, direction):
        src, dst, sport, dport, proto = pkt[0], pkt[1], pkt[2], pkt[3], pkt[4]
        if direction == FORWARD:
            self.src_ip, self.dst_ip = src, dst
            self.src_port, self.dst_port = sport, dport
        else:
            self.src_ip, self.dst_ip = dst, src
            self.src_port, self.dst_port = dport, sport
        self.protocol = proto

        self.start_t = t_us
        self.last_t = t_us
        self.last_fwd_t = 0
        self.last_bwd_t = 0

        self.fwd_len = RunningStats()
        self.bwd_len = RunningStats()
        self.all_len = RunningStats()
        self.flow_iat = RunningStats()
        self.fwd_iat = RunningStats()
        self.bwd_iat = RunningStats()

        self.fwd_hdr_bytes = 0
        self.bwd_hdr_bytes = 0
        self.min_seg_fwd = None
        self.fwd_act_data_pkts = 0

        self.fin = self.syn = self.rst = self.psh = 0
        self.ack = self.urg = self.ece = self.cwr = 0
        self.fwd_psh = self.bwd_psh = self.fwd_urg = self.bwd_urg = 0

        self.init_fwd_win = CFG.INIT_WIN_DEFAULT
        self.init_bwd_win = CFG.INIT_WIN_DEFAULT

        # Java seeds both active markers with the flow's first packet time,
        # which is what makes the first idle value sane (patch 9).
        self.start_active = t_us
        self.end_active = t_us
        self.active = RunningStats()
        self.idle = RunningStats()

        self.sf_last_t = t_us
        self.sf_count = 0

        self.fb_last_t = self.fb_start_tmp = 0
        self.fb_count = self.fb_count_tmp = 0
        self.fb_duration = self.fb_pkt_count = 0
        self.fb_size = self.fb_size_tmp = 0
        self.bb_last_t = self.bb_start_tmp = 0
        self.bb_count = self.bb_count_tmp = 0
        self.bb_duration = self.bb_pkt_count = 0
        self.bb_size = self.bb_size_tmp = 0

    # -- packet ingestion -------------------------------------------------
    def add_packet(self, pkt, t_us, direction):
        (_src, _dst, _sp, _dp, _proto, l4_hdr, payload, frame,
         flags, window, is_tcp) = pkt

        length = payload if CFG.PKT_LEN_MODE == "payload" else frame
        hdr = l4_hdr if CFG.HEADER_LEN_MODE == "l4" else 0

        npkts = self.fwd_len.n + self.bwd_len.n
        if npkts:  # not the first packet of the flow (patch 5)
            self._update_active_idle(t_us)
            self._update_subflow(t_us)
            self.flow_iat.add(t_us - self.last_t)

        self._update_bulk(t_us, payload, direction)

        if is_tcp:
            if flags & F_FIN:
                self.fin += 1
            if flags & F_SYN:
                self.syn += 1
            if flags & F_RST:
                self.rst += 1
            if flags & F_PSH:
                self.psh += 1
            if flags & F_ACK:
                self.ack += 1
            if flags & F_URG:
                self.urg += 1
            if flags & F_ECE:
                self.ece += 1
            if flags & F_CWR:
                self.cwr += 1

        if direction == FORWARD:
            if self.fwd_len.n:
                self.fwd_iat.add(t_us - self.last_fwd_t)
            elif is_tcp:
                self.init_fwd_win = window
            self.fwd_len.add(length)
            self.fwd_hdr_bytes += hdr
            if self.min_seg_fwd is None or l4_hdr < self.min_seg_fwd:
                self.min_seg_fwd = l4_hdr
            if payload >= 1:
                self.fwd_act_data_pkts += 1
            if is_tcp:
                if flags & F_PSH:
                    self.fwd_psh += 1
                if flags & F_URG:
                    self.fwd_urg += 1
            self.last_fwd_t = t_us
        else:
            if self.bwd_len.n:
                self.bwd_iat.add(t_us - self.last_bwd_t)
            elif is_tcp:
                self.init_bwd_win = window
            self.bwd_len.add(length)
            self.bwd_hdr_bytes += hdr
            if payload >= 1:
                pass  # Java tracks act_data_pkt for the forward side only
            if is_tcp:
                if flags & F_PSH:
                    self.bwd_psh += 1
                if flags & F_URG:
                    self.bwd_urg += 1
            self.last_bwd_t = t_us

        self.all_len.add(length)
        self.last_t = t_us

    def _update_active_idle(self, t_us):
        """Java BasicFlow.updateActiveIdleTime, called once per packet.

        The Java tool reaches this from two places and passes a duration in
        one of them; we call it once with the real timestamp (patch 9).
        """
        thr = int(CFG.ACTIVITY_TIMEOUT * US)
        if (t_us - self.end_active) > thr:
            dur = self.end_active - self.start_active
            if dur > 0:
                self.active.add(dur)
            self.idle.add(t_us - self.end_active)
            self.start_active = t_us
            self.end_active = t_us
        else:
            self.end_active = t_us

    def _update_subflow(self, t_us):
        if (t_us - self.sf_last_t) > int(CFG.CLUMP_TIMEOUT * US):
            self.sf_count += 1
        self.sf_last_t = t_us

    def _update_bulk(self, t_us, payload, direction):
        if payload <= 0:
            return
        clump = int(CFG.CLUMP_TIMEOUT * US)
        if direction == FORWARD:
            if self.bb_last_t > self.fb_start_tmp:
                self.fb_start_tmp = 0
            if self.fb_start_tmp == 0:
                self.fb_start_tmp = t_us
                self.fb_last_t = t_us
                self.fb_count_tmp = 1
                self.fb_size_tmp = payload
            elif (t_us - self.fb_last_t) > clump:
                self.fb_start_tmp = t_us
                self.fb_last_t = t_us
                self.fb_count_tmp = 1
                self.fb_size_tmp = payload
            else:
                self.fb_count_tmp += 1
                self.fb_size_tmp += payload
                if self.fb_count_tmp == CFG.BULK_BOUND:
                    self.fb_count += 1
                    self.fb_pkt_count += self.fb_count_tmp
                    self.fb_size += self.fb_size_tmp
                    self.fb_duration += t_us - self.fb_start_tmp
                elif self.fb_count_tmp > CFG.BULK_BOUND:
                    self.fb_pkt_count += 1
                    self.fb_size += payload
                    self.fb_duration += t_us - self.fb_last_t
                self.fb_last_t = t_us
        else:
            if self.fb_last_t > self.bb_start_tmp:
                self.bb_start_tmp = 0
            if self.bb_start_tmp == 0:
                self.bb_start_tmp = t_us
                self.bb_last_t = t_us
                self.bb_count_tmp = 1
                self.bb_size_tmp = payload
            elif (t_us - self.bb_last_t) > clump:
                self.bb_start_tmp = t_us
                self.bb_last_t = t_us
                self.bb_count_tmp = 1
                self.bb_size_tmp = payload
            else:
                self.bb_count_tmp += 1
                self.bb_size_tmp += payload
                if self.bb_count_tmp == CFG.BULK_BOUND:
                    self.bb_count += 1
                    self.bb_pkt_count += self.bb_count_tmp
                    self.bb_size += self.bb_size_tmp
                    self.bb_duration += t_us - self.bb_start_tmp
                elif self.bb_count_tmp > CFG.BULK_BOUND:
                    self.bb_pkt_count += 1
                    self.bb_size += payload
                    self.bb_duration += t_us - self.bb_last_t
                self.bb_last_t = t_us

    # -- output -----------------------------------------------------------
    @property
    def packet_count(self):
        return self.fwd_len.n + self.bwd_len.n

    @property
    def duration_us(self):
        return self.last_t - self.start_t

    def get_data(self):
        """All 76 CICFlowMeter features under the pip-cicflowmeter key names."""
        if CFG.ADD_FINAL_ACTIVE_SEGMENT:
            tail = self.end_active - self.start_active
            if tail > 0:
                self.active.add(tail)

        dur = self.duration_us
        dur_s = dur / float(US)
        n_fwd, n_bwd = self.fwd_len.n, self.bwd_len.n
        n_tot = n_fwd + n_bwd
        byts = self.fwd_len.total + self.bwd_len.total

        if dur > 0:
            flow_byts_s = byts / dur_s
            flow_pkts_s = n_tot / dur_s
            fwd_pkts_s = n_fwd / dur_s
            bwd_pkts_s = n_bwd / dur_s
        else:
            # Zero-duration flow.  The Java tool divides by 0.0 and so writes
            # Infinity (or NaN when the numerator is 0 too).  We always emit
            # Infinity, so that any NaN in the output is a genuine bug and the
            # Part 2 NaN check stays meaningful.  02_verify.py reports the
            # zero-duration flow count separately.
            inf = float("inf")
            flow_byts_s = flow_pkts_s = fwd_pkts_s = bwd_pkts_s = inf

        sf = self.sf_count if CFG.SUBFLOW_MODE == "divide" else 0
        if CFG.SUBFLOW_MODE == "divide":
            sf_div = sf + 1  # 1 subflow plus one per >1 s gap
            sub_fwd_pkts = _div(n_fwd, sf_div)
            sub_bwd_pkts = _div(n_bwd, sf_div)
            sub_fwd_byts = _div(self.fwd_len.total, sf_div)
            sub_bwd_byts = _div(self.bwd_len.total, sf_div)
        else:
            sub_fwd_pkts, sub_bwd_pkts = n_fwd, n_bwd
            sub_fwd_byts, sub_bwd_byts = self.fwd_len.total, self.bwd_len.total

        fb_rate = (self.fb_size / (self.fb_duration / float(US))
                   if self.fb_duration > 0 else 0)
        bb_rate = (self.bb_size / (self.bb_duration / float(US))
                   if self.bb_duration > 0 else 0)

        fwd_mean = self.fwd_len.mean
        bwd_mean = self.bwd_len.mean

        return {
            "src_ip": self.src_ip,
            "dst_ip": self.dst_ip,
            "src_port": self.src_port,
            "dst_port": self.dst_port,
            "protocol": self.protocol,
            "timestamp": self.start_t,          # integer microseconds
            "flow_duration": dur,
            "tot_fwd_pkts": n_fwd,
            "tot_bwd_pkts": n_bwd,
            "totlen_fwd_pkts": self.fwd_len.total,
            "totlen_bwd_pkts": self.bwd_len.total,
            "fwd_pkt_len_max": self.fwd_len.maximum,
            "fwd_pkt_len_min": self.fwd_len.minimum,
            "fwd_pkt_len_mean": fwd_mean,
            "fwd_pkt_len_std": self.fwd_len.std,
            "bwd_pkt_len_max": self.bwd_len.maximum,
            "bwd_pkt_len_min": self.bwd_len.minimum,
            "bwd_pkt_len_mean": bwd_mean,
            "bwd_pkt_len_std": self.bwd_len.std,
            "flow_byts_s": flow_byts_s,
            "flow_pkts_s": flow_pkts_s,
            "flow_iat_mean": self.flow_iat.mean,
            "flow_iat_std": self.flow_iat.std,
            "flow_iat_max": self.flow_iat.maximum,
            "flow_iat_min": self.flow_iat.minimum,
            "fwd_iat_tot": self.fwd_iat.total,
            "fwd_iat_mean": self.fwd_iat.mean,
            "fwd_iat_std": self.fwd_iat.std,
            "fwd_iat_max": self.fwd_iat.maximum,
            "fwd_iat_min": self.fwd_iat.minimum,
            "bwd_iat_tot": self.bwd_iat.total,
            "bwd_iat_mean": self.bwd_iat.mean,
            "bwd_iat_std": self.bwd_iat.std,
            "bwd_iat_max": self.bwd_iat.maximum,
            "bwd_iat_min": self.bwd_iat.minimum,
            "fwd_psh_flags": self.fwd_psh,
            "bwd_psh_flags": self.bwd_psh,
            "fwd_urg_flags": self.fwd_urg,
            "bwd_urg_flags": self.bwd_urg,
            "fwd_header_len": self.fwd_hdr_bytes,
            "bwd_header_len": self.bwd_hdr_bytes,
            "fwd_pkts_s": fwd_pkts_s,
            "bwd_pkts_s": bwd_pkts_s,
            "pkt_len_min": self.all_len.minimum,
            "pkt_len_max": self.all_len.maximum,
            "pkt_len_mean": self.all_len.mean,
            "pkt_len_std": self.all_len.std,
            "pkt_len_var": self.all_len.var,
            "fin_flag_cnt": self.fin,
            "syn_flag_cnt": self.syn,
            "rst_flag_cnt": self.rst,
            "psh_flag_cnt": self.psh,
            "ack_flag_cnt": self.ack,
            "urg_flag_cnt": self.urg,
            "cwr_flag_count": self.cwr,     # real CWR bit, not Fwd URG
            "ece_flag_cnt": self.ece,
            "down_up_ratio": _div(n_bwd, n_fwd) if n_fwd else 0,
            "pkt_size_avg": (byts / n_tot) if n_tot else 0,
            "fwd_seg_size_avg": fwd_mean,
            "bwd_seg_size_avg": bwd_mean,
            "fwd_byts_b_avg": _div(self.fb_size, self.fb_count),
            "fwd_pkts_b_avg": _div(self.fb_pkt_count, self.fb_count),
            "fwd_blk_rate_avg": fb_rate,
            "bwd_byts_b_avg": _div(self.bb_size, self.bb_count),
            "bwd_pkts_b_avg": _div(self.bb_pkt_count, self.bb_count),
            "bwd_blk_rate_avg": bb_rate,
            "subflow_fwd_pkts": sub_fwd_pkts,
            "subflow_fwd_byts": sub_fwd_byts,
            "subflow_bwd_pkts": sub_bwd_pkts,
            "subflow_bwd_byts": sub_bwd_byts,
            "init_fwd_win_byts": self.init_fwd_win,
            "init_bwd_win_byts": self.init_bwd_win,
            "fwd_act_data_pkts": self.fwd_act_data_pkts,
            "fwd_seg_size_min": (self.min_seg_fwd
                                 if self.min_seg_fwd is not None else 0),
            "active_mean": self.active.mean,
            "active_std": self.active.std,
            "active_max": self.active.maximum,
            "active_min": self.active.minimum,
            "idle_mean": self.idle.mean,
            "idle_std": self.idle.std,
            "idle_max": self.idle.maximum,
            "idle_min": self.idle.minimum,
        }


# ==========================================================================
# Engine
# ==========================================================================
class FlowEngine:
    """Builds flows from a capture and hands each finished flow to a sink."""

    def __init__(self, sink, progress=None):
        self.flows = {}
        self.sink = sink
        self.progress = progress
        self.counters = PacketCounters()
        self.flows_written = 0
        self.single_packet_flows = 0
        self.zero_duration_flows = 0
        self.flows_dropped_single = 0
        self.last_packet_time_us = 0

    def _emit(self, flow):
        if (CFG.DROP_SINGLE_PACKET_FLOWS and flow.packet_count <= 1):
            self.flows_dropped_single += 1
            return
        if flow.packet_count <= 1:
            self.single_packet_flows += 1
        if flow.duration_us == 0:
            self.zero_duration_flows += 1
        self.sink(flow.get_data())
        self.flows_written += 1

    def run(self, path):
        linkname, records = open_capture(path)
        c = self.counters
        flows = self.flows
        flow_to = int(CFG.FLOW_TIMEOUT * US)
        idle_to = int(CFG.IDLE_TIMEOUT * US)
        sweep_every = CFG.SWEEP_EVERY_PACKETS
        prog_every = CFG.PROGRESS_EVERY_PACKETS
        last_t_us = 0

        for t_us, caplen, origlen, data in records:
            c.total += 1
            if prog_every and self.progress and c.total % prog_every == 0:
                self.progress(c.total, len(flows), self.flows_written)

            off = _l2_ipv4_offset(linkname, data, caplen)
            if off < 0:
                c.non_ipv4 += 1
                continue
            pkt = parse_ipv4(data, off, caplen, c)
            if pkt is None:
                c.non_ipv4 += 1
                continue
            c.ipv4 += 1
            last_t_us = t_us

            src, dst, sp, dp, proto = pkt[0], pkt[1], pkt[2], pkt[3], pkt[4]
            fkey = (src, dst, sp, dp, proto)
            flow = flows.get(fkey)
            direction = FORWARD
            if flow is None:
                rkey = (dst, src, dp, sp, proto)
                flow = flows.get(rkey)
                if flow is not None:
                    fkey = rkey
                    direction = REVERSE

            if flow is None:
                flows[fkey] = Flow(pkt, t_us, FORWARD)
                flows[fkey].add_packet(pkt, t_us, FORWARD)
                continue

            # Patch 7: timeouts are checked on this flow only.
            if ((t_us - flow.start_t) > flow_to
                    or (t_us - flow.last_t) > idle_to):
                del flows[fkey]
                self._emit(flow)
                nkey = (src, dst, sp, dp, proto)
                flows[nkey] = Flow(pkt, t_us, FORWARD)
                flows[nkey].add_packet(pkt, t_us, FORWARD)
                continue

            flow.add_packet(pkt, t_us, direction)

            # Patch 4: real TCP flags.  Patch 7: no full rescan here.
            tcp_flags = pkt[8]
            if pkt[10] and ((CFG.CLOSE_FLOW_ON_FIN and (tcp_flags & F_FIN))
                            or (CFG.CLOSE_FLOW_ON_RST and (tcp_flags & F_RST))):
                del flows[fkey]
                self._emit(flow)
                continue

            if sweep_every and c.total % sweep_every == 0:
                self._sweep(t_us, flow_to, idle_to)

        # Flush everything still open.
        for flow in list(flows.values()):
            self._emit(flow)
        flows.clear()
        self.last_packet_time_us = last_t_us
        return self.counters

    def _sweep(self, now_us, flow_to, idle_to):
        dead = [k for k, f in self.flows.items()
                if (now_us - f.start_t) > flow_to
                or (now_us - f.last_t) > idle_to]
        for k in dead:
            f = self.flows.pop(k)
            self._emit(f)

## 5. Locating the Kaggle file and deriving the output schema

In [ ]:
import csv
import glob
import os


def find_original_file(csv_dir=None):
    """Return the most likely original MSCAD file in CSV_ORIGINAL, or None."""
    csv_dir = csv_dir or CFG.CSV_ORIGINAL_DIR
    if not os.path.isdir(csv_dir):
        return None, []
    cands = []
    for ext in ("*.csv", "*.CSV", "*.xlsx", "*.XLSX", "*.xls"):
        cands.extend(glob.glob(os.path.join(csv_dir, ext)))
    cands = sorted(set(cands))
    if not cands:
        return None, []

    def score(p):
        base = os.path.basename(p).lower()
        s = 0
        if "mscad" in base:
            s -= 100
        if base.endswith(".csv"):
            s -= 10
        s += len(base)
        return s

    return sorted(cands, key=score)[0], cands


def read_header(path):
    """Read just the header row of a csv/xlsx file."""
    if path.lower().endswith((".xlsx", ".xls")):
        import pandas as pd
        return list(pd.read_excel(path, nrows=0).columns)
    with open(path, "r", newline="", encoding="utf-8-sig", errors="replace") as fh:
        for row in csv.reader(fh):
            return row
    return []


def clean_names(names):
    """Strip surrounding whitespace and quotes, as Part 3 requires."""
    return [str(n).strip().strip('"').strip("'").strip() for n in names]


class Schema:
    """The output column layout.

    features : [(display_name, internal_key), ...] in original-file order
    unknown  : original header names that could not be mapped
    source   : where the layout came from
    """

    def __init__(self, features, unknown, source, original_header=None):
        self.features = features
        self.unknown = unknown
        self.source = source
        self.original_header = original_header or []

    @property
    def feature_display(self):
        return [d for d, _k in self.features]

    @property
    def feature_keys(self):
        return [k for _d, k in self.features]

    @property
    def out_columns(self):
        return (list(CFG.IDENTIFIER_COLUMNS) + self.feature_display
                + [CFG.LABEL_COLUMN])

    def describe(self):
        lines = ["Output schema source: %s" % self.source,
                 "Feature columns: %d" % len(self.features)]
        if self.unknown:
            lines.append("UNMAPPED original columns (%d): %s"
                         % (len(self.unknown), ", ".join(map(str, self.unknown))))
        else:
            lines.append("UNMAPPED original columns: none")
        return "\n".join(lines)


def build_schema(path=None, verbose=True):
    """Build the output schema from the original file, or fall back to the
    canonical 76 CICFlowMeter features if the original cannot be read."""
    if path is None:
        path, _all = find_original_file()
    if path and os.path.isfile(path):
        header = clean_names(read_header(path))
        feats, idents, unknown = resolve_header(header)
        # Drop duplicate keys, keeping the first occurrence.
        seen, uniq = set(), []
        for disp, key in feats:
            if key in seen:
                unknown.append("%s (duplicate of %s)" % (disp, key))
                continue
            seen.add(key)
            uniq.append((disp, key))
        if uniq:
            return Schema(uniq, unknown,
                          "original header of %s" % os.path.basename(path),
                          header)
        if verbose:
            print("  WARNING: no feature columns recognised in %s" % path)
    if verbose:
        print("  WARNING: original MSCAD file not readable; falling back to "
              "the canonical 76 CICFlowMeter feature names.")
    return Schema(list(CANONICAL_FEATURES), [],
                  "fallback: canonical 76 CICFlowMeter features", [])

## 6. The pipeline

Each stage is defined then run. Definition cells print nothing; the run cells do the
work and print the tables.

### 6.1 Stage 00 — inspect the inputs  *(definition)*

In [ ]:
import json
import os
import sys


def stage_00():
    print("=" * 78)
    print("STAGE 00 -- input inspection")
    print("=" * 78)
    report = {}

    # ---- PCAP folder ----------------------------------------------------
    pcap_dir, how = CFG.resolve_pcap_dir()
    print("\n[1] PCAP folder")
    for cand in CFG.PCAP_DIR_CANDIDATES:
        print("    candidate : %s  -> %s"
              % (cand, "EXISTS" if os.path.isdir(cand) else "missing"))
    if pcap_dir is None:
        print("    RESULT    : NOT FOUND (searched %s)" % CFG.PCAP_SEARCH_ROOT)
        report["pcap_dir"] = None
    else:
        print("    USING     : %s" % pcap_dir)
        print("    found via : %s" % how)
        report["pcap_dir"] = pcap_dir
        report["pcap_dir_how"] = how
        entries = sorted(os.listdir(pcap_dir))
        print("    contents  : %d entries" % len(entries))
        listing = []
        for e in entries:
            full = os.path.join(pcap_dir, e)
            size = os.path.getsize(full) if os.path.isfile(full) else 0
            listing.append({"name": e, "bytes": size})
            print("        %-28s %14s bytes" % (e, "{:,}".format(size)))
        report["pcap_dir_contents"] = listing

        print("\n    capture -> file resolution:")
        resolved = {}
        for key, label, desc in CFG.CAPTURES:
            f = CFG.find_capture_file(pcap_dir, key)
            resolved[key] = f
            print("        %-7s %-14s %s"
                  % (key, label, f if f else "*** NOT FOUND ***"))
        report["capture_files"] = resolved
        missing = [k for k, v in resolved.items() if not v]
        if missing:
            print("    WARNING: no file found for: %s" % ", ".join(missing))

    # ---- Original CSV folder -------------------------------------------
    print("\n[2] Original Kaggle CSV folder")
    print("    path      : %s  -> %s"
          % (CFG.CSV_ORIGINAL_DIR,
             "EXISTS" if os.path.isdir(CFG.CSV_ORIGINAL_DIR) else "missing"))
    orig, all_cands = find_original_file()
    report["csv_original_dir"] = CFG.CSV_ORIGINAL_DIR
    report["csv_original_candidates"] = all_cands
    report["original_file"] = orig
    if all_cands:
        print("    contents  :")
        for c in all_cands:
            print("        %-40s %14s bytes"
                  % (os.path.basename(c), "{:,}".format(os.path.getsize(c))))
    if orig is None:
        print("    RESULT    : no MSCAD csv/xlsx found")
    else:
        print("    USING     : %s" % orig)
        describe_original(orig, report)

    # ---- Schema ---------------------------------------------------------
    print("\n[3] Output schema")
    schema = build_schema(orig)
    print("    " + schema.describe().replace("\n", "\n    "))
    report["schema_source"] = schema.source
    report["schema_feature_count"] = len(schema.features)
    report["schema_features"] = schema.feature_display
    report["schema_unmapped"] = schema.unknown

    # ---- Output folder --------------------------------------------------
    CFG.ensure_out_dirs()
    print("\n[4] Output folder")
    print("    %s (created if absent)" % CFG.OUT_DIR)
    print("    %s" % os.path.join(CFG.OUT_DIR, CFG.RAW_SUBDIR))
    print("    %s" % os.path.join(CFG.OUT_DIR, CFG.METRICS_SUBDIR))

    with open(CFG.metrics_path("00_inputs.json"), "w") as fh:
        json.dump(report, fh, indent=2, default=str)
    print("\nWrote %s" % CFG.metrics_path("00_inputs.json"))
    return 0


def describe_original(path, report):
    import pandas as pd
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, low_memory=False, skipinitialspace=True)
    raw_cols = list(df.columns)
    df.columns = clean_names(raw_cols)

    print("    shape     : %d rows x %d columns" % (df.shape[0], df.shape[1]))
    report["original_shape"] = list(df.shape)
    report["original_columns_raw"] = [str(c) for c in raw_cols]
    report["original_columns_clean"] = [str(c) for c in df.columns]

    print("    columns   :")
    for i, c in enumerate(df.columns, 1):
        print("        %3d. %s" % (i, c))

    lab = None
    for cand in ("Label", "label", "Labels"):
        if cand in df.columns:
            lab = cand
            break
    if lab is None:
        print("    WARNING: no Label column found")
        report["original_label_counts"] = {}
    else:
        counts = df[lab].value_counts(dropna=False)
        print("    Label counts (column %r):" % lab)
        total = int(counts.sum())
        for k, v in counts.items():
            print("        %-18s %8d  (%5.2f%%)" % (k, v, 100.0 * v / total))
        print("        %-18s %8d" % ("TOTAL", total))
        report["original_label_counts"] = {str(k): int(v)
                                           for k, v in counts.items()}
        report["original_label_column"] = lab

### 6.2 Stage 00 — run

Checks both input folders and lists them, searching for `pcapngtopcap` if the
configured paths are missing and reporting which one it used. Then prints the original
Kaggle file's shape, every column name, and its Label counts.

Reads only — it never modifies, moves or deletes anything in the input folders.

In [ ]:
stage_00()

### 6.3 Stage 01 — PCAP to CSV  *(definition)*

In [ ]:
import csv
import datetime
import json
import os
import sys
import time


def fmt_timestamp(t_us):
    secs, micros = divmod(int(t_us), US)
    if CFG.TIMESTAMP_UTC:
        dt = datetime.datetime.fromtimestamp(secs, datetime.timezone.utc)
    else:
        dt = datetime.datetime.fromtimestamp(secs)
    return dt.replace(microsecond=micros).strftime(CFG.TIMESTAMP_FORMAT)


def fmt_value(key, v):
    """CSV text for one feature value."""
    if isinstance(v, str):
        return v
    if v != v:                              # NaN
        return "NaN"
    if v == float("inf"):
        return "Infinity"
    if v == float("-inf"):
        return "-Infinity"
    if key in INTEGER_KEYS:
        return str(int(round(v)))
    if isinstance(v, int):
        return str(v)
    if v == int(v) and abs(v) < 1e15:
        return str(int(v))
    return "%.10g" % v


def progress_printer(capture, t0):
    def _p(packets, open_flows, written):
        el = time.time() - t0
        rate = packets / el if el > 0 else 0
        print("    [%s] %10s packets  %7d open flows  %8d flows written"
              "  %6.0fs  %7.0f pkt/s"
              % (capture, "{:,}".format(packets), open_flows, written,
                 el, rate), flush=True)
    return _p


def detect_attackers(counts):
    """Attackers = every Src IP whose total forward packets reaches
    ATTACKER_SHARE of the top sender's."""
    if not counts:
        return set(), []
    ranked = sorted(counts.items(), key=lambda kv: -kv[1])
    top = ranked[0][1]
    cutoff = CFG.ATTACKER_SHARE * top
    attackers = {ip for ip, c in ranked if c >= cutoff}
    return attackers, ranked


def process_capture(key, label, desc, path, schema, writer_factory):
    print("\n" + "-" * 78)
    print("CAPTURE %s  (%s)  -- %s" % (key, label, desc))
    print("  file: %s" % path)
    print("  size: %s bytes" % "{:,}".format(os.path.getsize(path)))
    is_normal = (key == "N-0")

    # ---------------- pass 1: attacker detection ------------------------
    attackers, ranked = set(), []
    pass1 = None
    if not is_normal:
        print("  pass 1/2: building flows to find attacker IPs ...")
        t0 = time.time()
        counts = {}

        def count_sink(d):
            ip = d["src_ip"]
            counts[ip] = counts.get(ip, 0) + d["tot_fwd_pkts"]

        eng = FlowEngine(count_sink, progress_printer(key + " p1", t0))
        eng.run(path)
        pass1 = {"flows": eng.flows_written,
                 "packets": eng.counters.total,
                 "seconds": round(time.time() - t0, 1)}
        attackers, ranked = detect_attackers(counts)
        top = ranked[0][1] if ranked else 0
        print("  top %d senders by total forward packets:"
              % CFG.ATTACKER_TOP_N_PRINT)
        for ip, c in ranked[:CFG.ATTACKER_TOP_N_PRINT]:
            share = (100.0 * c / top) if top else 0
            print("        %-16s %12s fwd pkts  (%6.2f%% of top)%s"
                  % (ip, "{:,}".format(c), share,
                     "   <-- ATTACKER" if ip in attackers else ""))
        print("  attackers (>= %.0f%% of top sender): %s"
              % (100 * CFG.ATTACKER_SHARE,
                 ", ".join(sorted(attackers)) or "(none)"))
    else:
        print("  normal capture: every flow is labelled %r, "
              "no attacker detection" % label)

    # ---------------- pass 2: write flows -------------------------------
    print("  pass %s: writing flows ..." % ("2/2" if not is_normal else "1/1"))
    t0 = time.time()
    out_path = CFG.raw_csv_path(key)
    feature_keys = schema.feature_keys
    rows = [0]
    label_counts = {}

    with open(out_path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh, lineterminator="\n")
        w.writerow(schema.out_columns)

        def sink(d):
            if is_normal:
                lab = label
            elif d["src_ip"] in attackers or d["dst_ip"] in attackers:
                lab = label
            else:
                lab = CFG.BENIGN_LABEL
            row = [d["src_ip"], d["dst_ip"], d["src_port"], d["dst_port"],
                   d["protocol"], fmt_timestamp(d["timestamp"])]
            row.extend(fmt_value(k, d[k]) for k in feature_keys)
            row.append(lab)
            w.writerow(row)
            rows[0] += 1
            label_counts[lab] = label_counts.get(lab, 0) + 1

        eng = FlowEngine(sink, progress_printer(key + " p2", t0))
        counters = eng.run(path)

    elapsed = time.time() - t0
    print("  wrote %s flows to %s in %.1fs"
          % ("{:,}".format(rows[0]), out_path, elapsed))
    print("  packets read            : %s" % "{:,}".format(counters.total))
    print("  IPv4 packets in flows   : %s" % "{:,}".format(counters.ipv4))
    print("  non-IPv4 (ARP/IPv6/...) : %s" % "{:,}".format(counters.non_ipv4))
    print("  non-first IP fragments  : %s" % "{:,}".format(counters.fragments))
    print("  IPv4 by protocol        : %s"
          % ", ".join("%d:%s" % (p, "{:,}".format(c))
                      for p, c in sorted(counters.by_proto.items())))
    print("  single-packet flows     : %s"
          % "{:,}".format(eng.single_packet_flows))
    print("  zero-duration flows     : %s"
          % "{:,}".format(eng.zero_duration_flows))
    print("  labels                  : %s"
          % ", ".join("%s=%s" % (k, "{:,}".format(v))
                      for k, v in sorted(label_counts.items())))

    return {
        "capture": key,
        "official_label": label,
        "description": desc,
        "file": path,
        "file_bytes": os.path.getsize(path),
        "flows": rows[0],
        "label_counts": label_counts,
        "attackers": sorted(attackers),
        "top_senders": [{"src_ip": ip, "fwd_pkts": c} for ip, c in ranked[:10]],
        "pass1": pass1,
        "seconds": round(elapsed, 1),
        "single_packet_flows": eng.single_packet_flows,
        "zero_duration_flows": eng.zero_duration_flows,
        "counters": counters.as_dict(),
        "out_csv": out_path,
    }


def stage_01(captures=None):
    CFG.ensure_out_dirs()
    print("=" * 78)
    print("STAGE 01 -- PCAP -> flow CSV")
    print("=" * 78)
    print("flow timeout %.0fs | idle timeout %.0fs | activity %.0fs | "
          "clump %.0fs" % (CFG.FLOW_TIMEOUT, CFG.IDLE_TIMEOUT,
                           CFG.ACTIVITY_TIMEOUT, CFG.CLUMP_TIMEOUT))
    print("pkt-len mode %r | header-len mode %r | std ddof %d | subflow %r"
          % (CFG.PKT_LEN_MODE, CFG.HEADER_LEN_MODE, CFG.STD_DDOF,
             CFG.SUBFLOW_MODE))
    print("time features in MICROSECONDS")

    pcap_dir, how = CFG.resolve_pcap_dir()
    if pcap_dir is None:
        print("\nFATAL: PCAP folder not found. Candidates tried:")
        for c in CFG.PCAP_DIR_CANDIDATES:
            print("    %s" % c)
        print("Also searched recursively under %s" % CFG.PCAP_SEARCH_ROOT)
        return 2
    print("\nPCAP folder: %s  (%s)" % (pcap_dir, how))

    schema = build_schema()
    print(schema.describe())
    if schema.unknown:
        print("NOTE: the unmapped columns above are NOT written to the output.")

    wanted = list(captures or [])
    captures = [c for c in CFG.CAPTURES if not wanted or c[0] in wanted]
    if wanted:
        unknown = set(wanted) - {c[0] for c in CFG.CAPTURES}
        if unknown:
            print("FATAL: unknown capture key(s): %s" % ", ".join(unknown))
            return 2

    results, failures = [], []
    t_all = time.time()
    for key, label, desc in captures:
        path = CFG.find_capture_file(pcap_dir, key)
        if path is None:
            msg = ("capture %s: no file named %s%s in %s"
                   % (key, key, "/".join(CFG.PCAP_EXTENSIONS), pcap_dir))
            print("\nERROR: " + msg)
            failures.append({"capture": key, "error": msg})
            continue
        try:
            results.append(process_capture(key, label, desc, path, schema,
                                           None))
        except Exception as exc:  # noqa: BLE001 - surfaced, never swallowed
            import traceback
            print("\nERROR while processing %s:" % key)
            traceback.print_exc()
            failures.append({"capture": key,
                             "error": "%s: %s" % (type(exc).__name__, exc)})

    out = {
        "pcap_dir": pcap_dir,
        "pcap_dir_how": how,
        "schema_source": schema.source,
        "schema_features": schema.feature_display,
        "schema_unmapped": schema.unknown,
        "config": {
            "flow_timeout_s": CFG.FLOW_TIMEOUT,
            "idle_timeout_s": CFG.IDLE_TIMEOUT,
            "activity_timeout_s": CFG.ACTIVITY_TIMEOUT,
            "clump_timeout_s": CFG.CLUMP_TIMEOUT,
            "pkt_len_mode": CFG.PKT_LEN_MODE,
            "header_len_mode": CFG.HEADER_LEN_MODE,
            "std_ddof": CFG.STD_DDOF,
            "subflow_mode": CFG.SUBFLOW_MODE,
            "init_win_default": CFG.INIT_WIN_DEFAULT,
            "java_integer_division": CFG.JAVA_INTEGER_DIVISION,
            "drop_single_packet_flows": CFG.DROP_SINGLE_PACKET_FLOWS,
            "add_final_active_segment": CFG.ADD_FINAL_ACTIVE_SEGMENT,
            "attacker_share": CFG.ATTACKER_SHARE,
            "close_flow_on_fin": CFG.CLOSE_FLOW_ON_FIN,
            "close_flow_on_rst": CFG.CLOSE_FLOW_ON_RST,
        },
        "captures": results,
        "failures": failures,
        "total_seconds": round(time.time() - t_all, 1),
    }
    name = ("01_extraction.json" if not wanted
            else "01_extraction_%s.json" % "_".join(wanted))
    with open(CFG.metrics_path(name), "w") as fh:
        json.dump(out, fh, indent=2, default=str)

    print("\n" + "=" * 78)
    print("STAGE 01 SUMMARY  (%.1fs total)" % out["total_seconds"])
    print("=" * 78)
    print("%-8s %10s %12s %12s  %s"
          % ("capture", "flows", "pkts read", "IPv4 flows", "labels"))
    for r in results:
        print("%-8s %10s %12s %12s  %s"
              % (r["capture"], "{:,}".format(r["flows"]),
                 "{:,}".format(r["counters"]["packets_read"]),
                 "{:,}".format(r["counters"]["ipv4_packets"]),
                 ", ".join("%s=%d" % kv
                           for kv in sorted(r["label_counts"].items()))))
    if failures:
        print("\nFAILURES (%d):" % len(failures))
        for f in failures:
            print("    %s: %s" % (f["capture"], f["error"]))
    print("\nMetrics: %s" % CFG.metrics_path(name))
    return 1 if failures else 0

### 6.4 Stage 01a — N-0 only, the quick test

**Check this before the full run:**

- `packets read` minus `IPv4 packets in flows` must equal the non-IPv4 count and nothing else
- the feature column count matches your Kaggle file (66)
- no column reported as unmappable
- `Flow Duration` looks like microseconds — tens of thousands for a short session

In [ ]:
stage_01(["N-0"])

Peek at the N-0 rows:

In [ ]:
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

_p = CFG.raw_csv_path("N-0")
if os.path.isfile(_p):
    _df = pd.read_csv(_p)
    print("shape:", _df.shape)
    _cols = ["Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol", "Timestamp",
             "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts",
             "Fwd Pkt Len Min", "Fwd Pkt Len Max", "Flow IAT Mean",
             "Fwd Header Len", "Init Bwd Win Byts", "Label"]
    print(_df[[c for c in _cols if c in _df.columns]].head(10).to_string())
    print("\npackets accounted for in flows:",
          int(_df["Tot Fwd Pkts"].sum() + _df["Tot Bwd Pkts"].sum()))
    print("protocols (6=TCP, 17=UDP, 1=ICMP):", dict(_df["Protocol"].value_counts()))
else:
    print("not found yet:", _p)

### 6.5 Stage 01b — all six captures

Progress prints every 100,000 packets. Each attack capture is read twice — once to find
the attacker IPs, once to write labelled flows — so nothing is held in memory. The top
5 senders per capture are printed so you can check the attacker detection yourself.

Expect a couple of minutes, not 10–20: a 704,000-packet capture measured 19.5 s.

In [ ]:
stage_01()

### 6.6 Stage 02 — verify the extraction  *(definition)*

In [ ]:
import json
import os
import struct
import subprocess
import sys

import numpy as np
import pandas as pd


MINMAX_PAIRS = [
    ("fwd_pkt_len_min", "fwd_pkt_len_max"),
    ("bwd_pkt_len_min", "bwd_pkt_len_max"),
    ("pkt_len_min", "pkt_len_max"),
    ("flow_iat_min", "flow_iat_max"),
    ("fwd_iat_min", "fwd_iat_max"),
    ("bwd_iat_min", "bwd_iat_max"),
    ("active_min", "active_max"),
    ("idle_min", "idle_max"),
]
# These two are -1 by design when the direction carries no TCP packet.
NEGATIVE_ALLOWED = {"init_fwd_win_byts", "init_bwd_win_byts"}


def tshark_ipv4_count(path):
    """Independent IPv4 packet count via tshark. Returns (count, note)."""
    exe = CFG.TSHARK_EXE
    if not os.path.isfile(exe):
        return None, "tshark not found at %s" % exe
    cmd = [exe, "-r", path, "-Y", "ip", "-T", "fields", "-e", "frame.number"]
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
    except Exception as exc:  # noqa: BLE001
        return None, "tshark failed: %s: %s" % (type(exc).__name__, exc)
    if p.returncode != 0:
        return None, "tshark exit %d: %s" % (p.returncode,
                                             (p.stderr or "").strip()[:300])
    n = sum(1 for line in p.stdout.splitlines() if line.strip())
    return n, "tshark -Y ip"


def second_opinion_ipv4_count(path):
    """A deliberately separate IPv4 packet counter.

    Written against the raw file bytes with its own record walk, independent
    of mscad_flowmeter's reader, and with no third-party imports at all -- so
    this check still runs on a stock Anaconda install with no scapy and no
    Wireshark. If it and the flow totals agree, no packet was lost or
    double-counted.
    """
    try:
        with open(path, "rb") as fh:
            gh = fh.read(24)
            if len(gh) < 24:
                return None, "file shorter than a pcap header"
            magic = gh[:4]
            if magic in (b"\xd4\xc3\xb2\xa1", b"\x4d\x3c\xb2\xa1"):
                end = "<"
            elif magic in (b"\xa1\xb2\xc3\xd4", b"\xa1\xb2\x3c\x4d"):
                end = ">"
            elif magic[:4] == b"\x0a\x0d\x0d\x0a":
                return None, ("pcapng not supported by this counter; use "
                              "tshark, or convert with editcap -F pcap")
            else:
                return None, "unrecognised pcap magic %r" % magic
            linktype = struct.unpack(end + "I", gh[20:24])[0]
            if linktype not in (0, 1, 12, 101, 113, 228, 276):
                return None, "unsupported datalink type %d" % linktype

            hdr = struct.Struct(end + "IIII")
            n = 0
            total = 0
            while True:
                rh = fh.read(16)
                if len(rh) < 16:
                    break
                _s, _u, caplen, _orig = hdr.unpack(rh)
                data = fh.read(caplen)
                if len(data) < caplen:
                    break   # truncated final record
                total += 1
                if linktype == 1:
                    if len(data) < 14:
                        continue
                    et = (data[12] << 8) | data[13]
                    base = 14
                    while et in (0x8100, 0x88A8, 0x9100) and len(data) >= base + 4:
                        et = (data[base + 2] << 8) | data[base + 3]
                        base += 4
                    if et == 0x0800 and len(data) > base \
                            and (data[base] >> 4) == 4:
                        n += 1
                elif linktype in (12, 101, 228):
                    if data and (data[0] >> 4) == 4:
                        n += 1
                elif linktype == 113:
                    if len(data) >= 16 \
                            and ((data[14] << 8) | data[15]) == 0x0800:
                        n += 1
                elif linktype == 276:
                    if len(data) >= 20 \
                            and ((data[0] << 8) | data[1]) == 0x0800:
                        n += 1
                elif linktype == 0:
                    if len(data) >= 4 and (
                            struct.unpack("<I", data[0:4])[0] == 2
                            or struct.unpack(">I", data[0:4])[0] == 2):
                        n += 1
    except Exception as exc:  # noqa: BLE001 - surfaced in the table, not hidden
        return None, "%s: %s" % (type(exc).__name__, exc)
    return n, "independent raw-bytes counter (no scapy, no tshark)"


def load_raw(path, schema):
    """Read a regenerated CSV, keeping Infinity as numpy inf."""
    df = pd.read_csv(path, low_memory=False, skipinitialspace=True,
                     na_values=["", "nan", "NaN"], keep_default_na=False)
    df.columns = clean_names(df.columns)
    # Normalise the textual infinities the writer emits.
    for col in schema.feature_display:
        if col in df.columns and df[col].dtype == object:
            df[col] = pd.to_numeric(
                df[col].replace({"Infinity": np.inf, "-Infinity": -np.inf,
                                 "inf": np.inf, "-inf": -np.inf}),
                errors="coerce")
    return df


def verify_capture(key, label, pcap_path, schema, extraction):
    res = {"capture": key, "official_label": label, "problems": []}
    csv_path = CFG.raw_csv_path(key)
    if not os.path.isfile(csv_path):
        res["problems"].append("missing regenerated CSV %s" % csv_path)
        return res

    df = load_raw(csv_path, schema)
    res["flows"] = int(len(df))
    disp = {k: d for d, k in schema.features}

    # ---- packet accounting ---------------------------------------------
    fwd_col = disp.get("tot_fwd_pkts")
    bwd_col = disp.get("tot_bwd_pkts")
    if fwd_col and bwd_col:
        in_flows = int(df[fwd_col].sum() + df[bwd_col].sum())
    else:
        in_flows = None
        res["problems"].append("Tot Fwd/Bwd Pkts not in schema; cannot "
                               "verify packet conservation")
    res["ipv4_in_flows"] = in_flows
    res["packets_read"] = (extraction or {}).get(
        "counters", {}).get("packets_read")
    res["non_ipv4"] = (extraction or {}).get(
        "counters", {}).get("non_ipv4_packets")
    res["ipv4_parsed"] = (extraction or {}).get(
        "counters", {}).get("ipv4_packets")

    if pcap_path and os.path.isfile(pcap_path):
        res["tshark_ipv4"], res["tshark_note"] = tshark_ipv4_count(pcap_path)
        res["second_ipv4"], res["second_note"] = \
            second_opinion_ipv4_count(pcap_path)
    else:
        res["tshark_ipv4"] = res["second_ipv4"] = None
        res["tshark_note"] = res["second_note"] = "pcap not available"

    for name in ("tshark_ipv4", "second_ipv4", "ipv4_parsed"):
        v = res.get(name)
        if v is not None and in_flows is not None and v != in_flows:
            res["problems"].append(
                "%s = %d but flows account for %d IPv4 packets (diff %+d)"
                % (name, v, in_flows, v - in_flows))

    # ---- duplicate flows ------------------------------------------------
    dup_cols = [c for c in ["Src IP", "Dst IP", "Src Port", "Dst Port",
                            "Protocol", "Timestamp"] if c in df.columns]
    res["duplicate_flows"] = int(df.duplicated(subset=dup_cols).sum()) \
        if dup_cols else None
    if res["duplicate_flows"]:
        res["problems"].append("%d flows share a 5-tuple and start timestamp"
                               % res["duplicate_flows"])

    # ---- value sanity ---------------------------------------------------
    feats = [c for c in schema.feature_display if c in df.columns]
    block = df[feats]
    res["nan_cells"] = int(block.isna().sum().sum())
    inf_mask = np.isinf(block.to_numpy(dtype="float64"))
    res["inf_cells"] = int(inf_mask.sum())
    res["inf_by_column"] = {c: int(n) for c, n in
                            zip(feats, inf_mask.sum(axis=0)) if n}
    res["zero_duration_flows"] = (extraction or {}).get("zero_duration_flows")
    res["single_packet_flows"] = (extraction or {}).get("single_packet_flows")
    if res["nan_cells"]:
        res["problems"].append("%d NaN cells" % res["nan_cells"])

    neg_cols = [c for c in feats
                if NAME_TO_KEY.get(normalise(c)) not in NEGATIVE_ALLOWED]
    negs = (df[neg_cols] < 0).sum()
    res["negative_cells"] = int(negs.sum())
    res["negative_by_column"] = {c: int(v) for c, v in negs.items() if v}
    if res["negative_cells"]:
        res["problems"].append("%d negative feature values" %
                               res["negative_cells"])
    res["init_win_minus_one"] = {
        disp[k]: int((df[disp[k]] == -1).sum())
        for k in NEGATIVE_ALLOWED if k in disp and disp[k] in df.columns}

    bad_minmax = {}
    for kmin, kmax in MINMAX_PAIRS:
        cmin, cmax = disp.get(kmin), disp.get(kmax)
        if cmin in df.columns and cmax in df.columns:
            n = int((df[cmin] > df[cmax]).sum())
            if n:
                bad_minmax["%s > %s" % (cmin, cmax)] = n
    res["minmax_violations"] = bad_minmax
    res["minmax_violation_total"] = int(sum(bad_minmax.values()))
    if bad_minmax:
        res["problems"].append("min > max in: %s" % bad_minmax)

    bad_ports = 0
    for col in ("Src Port", "Dst Port"):
        if col in df.columns:
            p = pd.to_numeric(df[col], errors="coerce")
            bad_ports += int(((p < 0) | (p > 65535) | p.isna()).sum())
    res["invalid_ports"] = bad_ports
    if bad_ports:
        res["problems"].append("%d invalid port values" % bad_ports)

    bad_proto = 0
    if "Protocol" in df.columns:
        pr = pd.to_numeric(df["Protocol"], errors="coerce")
        bad_proto = int(((pr < 0) | (pr > 255) | pr.isna()).sum())
    res["invalid_protocol"] = bad_proto
    if bad_proto:
        res["problems"].append("%d invalid protocol values" % bad_proto)

    # ---- duplicate feature vectors --------------------------------------
    dup_feat = int(block.duplicated().sum())
    res["duplicate_feature_rows"] = dup_feat
    res["unique_feature_rows"] = int(len(block) - dup_feat)

    res["label_counts"] = {str(k): int(v) for k, v in
                           df[CFG.LABEL_COLUMN].value_counts().items()} \
        if CFG.LABEL_COLUMN in df.columns else {}
    res["protocol_counts"] = {str(k): int(v) for k, v in
                              df["Protocol"].value_counts().items()} \
        if "Protocol" in df.columns else {}
    return res


def stage_02(captures=None):
    CFG.ensure_out_dirs()
    print("=" * 78)
    print("STAGE 02 -- verification of the extraction (Part 2)")
    print("=" * 78)

    extraction = {}
    mp = CFG.metrics_path("01_extraction.json")
    if os.path.isfile(mp):
        with open(mp) as fh:
            for c in json.load(fh).get("captures", []):
                extraction[c["capture"]] = c
    else:
        print("NOTE: %s not found; packets-read figures will be blank." % mp)

    schema = build_schema()
    pcap_dir, _how = CFG.resolve_pcap_dir()

    wanted = list(captures or [])
    results = []
    for key, label, _desc in CFG.CAPTURES:
        if wanted and key not in wanted:
            continue
        if not os.path.isfile(CFG.raw_csv_path(key)):
            print("\n%s: no regenerated CSV yet, skipping" % key)
            continue
        print("\nverifying %s ..." % key, flush=True)
        pcap = CFG.find_capture_file(pcap_dir, key) if pcap_dir else None
        results.append(verify_capture(key, label, pcap, schema,
                                      extraction.get(key)))

    # ---- tables ---------------------------------------------------------
    print("\n" + "=" * 110)
    print("TABLE 2.1  packet accounting   (flow total must equal the "
          "independent counts)")
    print("=" * 110)
    hdr = ("%-8s %12s %14s %14s %14s %12s %10s"
           % ("capture", "pkts read", "IPv4 in flows", "tshark IPv4",
              "2nd-opinion", "non-IPv4", "match"))
    print(hdr)
    print("-" * 110)
    for r in results:
        ok = all(r.get(k) in (None, r.get("ipv4_in_flows"))
                 for k in ("tshark_ipv4", "second_ipv4", "ipv4_parsed"))
        print("%-8s %12s %14s %14s %14s %12s %10s"
              % (r["capture"],
                 _n(r.get("packets_read")), _n(r.get("ipv4_in_flows")),
                 _n(r.get("tshark_ipv4")), _n(r.get("second_ipv4")),
                 _n(r.get("non_ipv4")), "YES" if ok else "NO"))
    notes = {r["capture"]: r.get("tshark_note") for r in results
             if r.get("tshark_ipv4") is None}
    for k, v in notes.items():
        print("    %s: tshark count unavailable -- %s" % (k, v))

    print("\n" + "=" * 110)
    print("TABLE 2.2  data sanity")
    print("=" * 110)
    print("%-8s %9s %7s %7s %7s %8s %8s %9s %10s %12s"
          % ("capture", "flows", "dupes", "NaN", "inf", "neg", "min>max",
             "badport", "1-pkt flws", "dup feat rows"))
    print("-" * 110)
    for r in results:
        print("%-8s %9s %7s %7s %7s %8s %8s %9s %10s %12s"
              % (r["capture"], _n(r.get("flows")),
                 _n(r.get("duplicate_flows")), _n(r.get("nan_cells")),
                 _n(r.get("inf_cells")), _n(r.get("negative_cells")),
                 _n(r.get("minmax_violation_total")),
                 _n(r.get("invalid_ports")),
                 _n(r.get("single_packet_flows")),
                 _n(r.get("duplicate_feature_rows"))))
    print("\n(inf cells are the zero-duration flows' Flow Byts/s, Flow "
          "Pkts/s, Fwd Pkts/s, Bwd Pkts/s -- normal CICFlowMeter behaviour.)")
    for r in results:
        if r.get("inf_by_column"):
            print("    %-8s inf by column: %s"
                  % (r["capture"], r["inf_by_column"]))
        if r.get("zero_duration_flows") is not None:
            print("    %-8s zero-duration flows: %s"
                  % (r["capture"], _n(r["zero_duration_flows"])))

    print("\n" + "=" * 110)
    print("TABLE 2.3  labels and protocols per capture")
    print("=" * 110)
    for r in results:
        print("%-8s labels: %-55s protocols: %s"
              % (r["capture"],
                 ", ".join("%s=%s" % (k, _n(v))
                           for k, v in sorted(r["label_counts"].items())),
                 ", ".join("%s=%s" % (k, _n(v))
                           for k, v in sorted(r["protocol_counts"].items()))))

    problems = [(r["capture"], p) for r in results for p in r["problems"]]
    print("\n" + "=" * 110)
    if problems:
        print("PROBLEMS FOUND (%d)" % len(problems))
        for cap, p in problems:
            print("    %-8s %s" % (cap, p))
    else:
        print("NO PROBLEMS FOUND: packet counts reconcile, no duplicate "
              "flows, no NaN, no negatives, no min>max, all ports valid.")
    print("=" * 110)

    with open(CFG.metrics_path("02_verification.json"), "w") as fh:
        json.dump({"captures": results, "problem_count": len(problems)},
                  fh, indent=2, default=str)
    print("\nMetrics: %s" % CFG.metrics_path("02_verification.json"))
    return 0


def _n(v):
    if v is None:
        return "-"
    if isinstance(v, (int, np.integer)):
        return "{:,}".format(int(v))
    return str(v)

### 6.7 Stage 02 — run

The first table is the one that matters: `IPv4 in flows` must equal the independent
counts **exactly**. If it does, no packet was lost or double-counted.

In the sanity table everything should be 0 except `inf` (zero-duration flows' rate
features — normal CICFlowMeter behaviour), `1-pkt flws` and `dup feat rows`, which are
expected in scans and floods and are counted rather than dropped.

In [ ]:
stage_02()

### 6.8 Stage 03 — compare with the Kaggle file  *(definition)*

In [ ]:
import json
import os
import sys
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as sstats


MAX_KS_SAMPLE = 200_000   # cap per side so ks_2samp stays tractable
RNG = np.random.default_rng(20260101)


# --------------------------------------------------------------------------
def to_numeric_block(df, cols):
    out = pd.DataFrame(index=df.index)
    for c in cols:
        s = df[c]
        if s.dtype == object:
            s = s.replace({"Infinity": np.inf, "-Infinity": -np.inf,
                           "inf": np.inf, "-inf": -np.inf, "": np.nan})
        out[c] = pd.to_numeric(s, errors="coerce")
    return out


def load_original(path):
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, low_memory=False, skipinitialspace=True)
    df.columns = clean_names(df.columns)
    return df


def load_regenerated(schema):
    frames = []
    for key, _label, _desc in CFG.CAPTURES:
        p = CFG.raw_csv_path(key)
        if not os.path.isfile(p):
            print("  NOTE: %s missing, not included" % p)
            continue
        d = pd.read_csv(p, low_memory=False, skipinitialspace=True,
                        keep_default_na=False,
                        na_values=["", "nan", "NaN"])
        d.columns = clean_names(d.columns)
        d["Source_Capture"] = key
        d["_row"] = np.arange(len(d))
        frames.append(d)
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)


def ks(a, b):
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return float("nan"), float("nan"), len(a), len(b)
    if len(a) > MAX_KS_SAMPLE:
        a = RNG.choice(a, MAX_KS_SAMPLE, replace=False)
    if len(b) > MAX_KS_SAMPLE:
        b = RNG.choice(b, MAX_KS_SAMPLE, replace=False)
    r = sstats.ks_2samp(a, b)
    return float(r.statistic), float(r.pvalue), len(a), len(b)


# --------------------------------------------------------------------------
def stage_03():
    CFG.ensure_out_dirs()
    print("=" * 78)
    print("STAGE 03 -- comparison with the original Kaggle MSCAD file")
    print("=" * 78)

    orig_path, _cands = find_original_file()
    if orig_path is None:
        print("FATAL: no original MSCAD csv/xlsx found in %s"
              % CFG.CSV_ORIGINAL_DIR)
        return 2
    print("original    : %s" % orig_path)

    schema = build_schema(orig_path)
    orig = load_original(orig_path)
    regen = load_regenerated(schema)
    if regen is None:
        print("FATAL: no regenerated CSVs in %s"
              % os.path.join(CFG.OUT_DIR, CFG.RAW_SUBDIR))
        return 2
    print("regenerated : %d flows from %d captures"
          % (len(regen), regen["Source_Capture"].nunique()))

    out = {"original_file": orig_path,
           "original_shape": list(orig.shape),
           "regenerated_shape": [int(len(regen)),
                                 int(len(schema.feature_display))]}

    # ---- 3.1 schema -----------------------------------------------------
    print("\n" + "=" * 78)
    print("3.1  SCHEMA")
    print("=" * 78)
    feats = schema.feature_display
    orig_label_col = next((c for c in ("Label", "label", "Labels")
                           if c in orig.columns), None)
    orig_feats = [c for c in orig.columns if c != orig_label_col]
    missing_in_regen = [c for c in orig_feats if c not in regen.columns]
    extra_in_regen = [c for c in feats if c not in orig.columns]
    same_order = (list(feats) ==
                  [c for c in orig_feats if c in feats])
    print("original feature columns    : %d" % len(orig_feats))
    print("regenerated feature columns : %d" % len(feats))
    print("names and order identical   : %s" % ("YES" if
          (not missing_in_regen and not extra_in_regen and same_order)
          else "NO"))
    if missing_in_regen:
        print("  in original but NOT regenerated (%d): %s"
              % (len(missing_in_regen), ", ".join(map(str, missing_in_regen))))
    if extra_in_regen:
        print("  regenerated but NOT in original (%d): %s"
              % (len(extra_in_regen), ", ".join(map(str, extra_in_regen))))
    if schema.unknown:
        print("  original columns that could not be mapped to a "
              "CICFlowMeter feature (%d): %s"
              % (len(schema.unknown), ", ".join(map(str, schema.unknown))))
    out["schema"] = {
        "original_feature_columns": [str(c) for c in orig_feats],
        "regenerated_feature_columns": list(feats),
        "missing_in_regenerated": [str(c) for c in missing_in_regen],
        "extra_in_regenerated": [str(c) for c in extra_in_regen],
        "same_names_and_order": bool(not missing_in_regen
                                     and not extra_in_regen and same_order),
        "unmapped_original_columns": [str(c) for c in schema.unknown],
    }

    common = [c for c in feats if c in orig.columns]
    print("common features used for the comparisons below: %d" % len(common))
    out["common_features"] = common

    if orig_label_col is None:
        print("FATAL: original file has no Label column")
        return 2

    # ---- 3.2 rows per label ---------------------------------------------
    print("\n" + "=" * 78)
    print("3.2  ROWS PER LABEL")
    print("=" * 78)
    oc = orig[orig_label_col].astype(str).value_counts()
    rc = regen[CFG.LABEL_COLUMN].astype(str).value_counts()
    labels = sorted(set(oc.index) | set(rc.index))
    print("%-18s %12s %14s %12s %10s" % ("label", "Kaggle", "regenerated",
                                         "difference", "diff %"))
    print("-" * 70)
    rows32 = []
    for lab in labels:
        o = int(oc.get(lab, 0))
        r = int(rc.get(lab, 0))
        d = r - o
        pct = (100.0 * d / o) if o else float("nan")
        rows32.append({"label": lab, "kaggle": o, "regenerated": r,
                       "difference": d, "diff_pct": pct})
        print("%-18s %12s %14s %12s %10s"
              % (lab, "{:,}".format(o), "{:,}".format(r),
                 "{:+,}".format(d),
                 "-" if o == 0 else "%+.1f%%" % pct))
    print("-" * 70)
    print("%-18s %12s %14s %12s"
          % ("TOTAL", "{:,}".format(int(oc.sum())),
             "{:,}".format(int(rc.sum())),
             "{:+,}".format(int(rc.sum() - oc.sum()))))
    only_o = sorted(set(oc.index) - set(rc.index))
    only_r = sorted(set(rc.index) - set(oc.index))
    if only_o:
        print("labels only in Kaggle      : %s" % ", ".join(only_o))
    if only_r:
        print("labels only in regenerated : %s" % ", ".join(only_r))
    out["rows_per_label"] = rows32
    out["labels_only_kaggle"] = only_o
    out["labels_only_regenerated"] = only_r

    shared_labels = [l for l in labels
                     if oc.get(l, 0) > 0 and rc.get(l, 0) > 0]

    # ---- 3.3 units ------------------------------------------------------
    print("\n" + "=" * 78)
    print("3.3  UNITS CHECK  (both files should be in MICROSECONDS)")
    print("=" * 78)
    unit_cols = [c for c in ("Flow Duration", "Flow IAT Mean") if c in common]
    rows33 = []
    if not unit_cols:
        print("Flow Duration / Flow IAT Mean not available in both files.")
    else:
        print("%-18s %-16s %16s %16s %10s %s"
              % ("label", "feature", "Kaggle median", "regen median",
                 "ratio", "verdict"))
        print("-" * 100)
        for lab in shared_labels:
            ob = to_numeric_block(orig[orig[orig_label_col].astype(str) == lab],
                                  unit_cols)
            rb = to_numeric_block(regen[regen[CFG.LABEL_COLUMN].astype(str)
                                        == lab], unit_cols)
            for c in unit_cols:
                om = float(np.nanmedian(ob[c].to_numpy(dtype="float64"))) \
                    if len(ob) else float("nan")
                rm = float(np.nanmedian(rb[c].to_numpy(dtype="float64"))) \
                    if len(rb) else float("nan")
                ratio = (om / rm) if (rm not in (0,) and np.isfinite(rm)
                                      and np.isfinite(om) and rm != 0) \
                    else float("nan")
                verdict = "ok"
                if np.isfinite(ratio):
                    if 1e5 < ratio < 1e7:
                        verdict = "*** Kaggle looks 10^6 LARGER (unit mismatch)"
                    elif 1e-7 < ratio < 1e-5:
                        verdict = "*** regen looks 10^6 LARGER (unit mismatch)"
                rows33.append({"label": lab, "feature": c,
                               "kaggle_median": om, "regen_median": rm,
                               "ratio": ratio, "verdict": verdict})
                print("%-18s %-16s %16.3f %16.3f %10s %s"
                      % (lab, c, om, rm,
                         "-" if not np.isfinite(ratio) else "%.4g" % ratio,
                         verdict))
    out["units_check"] = rows33

    # ---- 3.4 per-feature distributions ---------------------------------
    print("\n" + "=" * 78)
    print("3.4  PER-FEATURE DISTRIBUTION COMPARISON PER LABEL")
    print("=" * 78)
    per_label = {}
    for lab in shared_labels:
        ob = to_numeric_block(orig[orig[orig_label_col].astype(str) == lab],
                              common)
        rb = to_numeric_block(regen[regen[CFG.LABEL_COLUMN].astype(str) == lab],
                              common)
        recs = []
        for c in common:
            a = ob[c].to_numpy(dtype="float64")
            b = rb[c].to_numpy(dtype="float64")
            st, pv, na, nb = ks(a, b)
            with np.errstate(all="ignore"):
                recs.append({
                    "feature": c,
                    "kaggle_mean": float(np.nanmean(a[np.isfinite(a)]))
                    if np.isfinite(a).any() else float("nan"),
                    "regen_mean": float(np.nanmean(b[np.isfinite(b)]))
                    if np.isfinite(b).any() else float("nan"),
                    "kaggle_median": float(np.nanmedian(a[np.isfinite(a)]))
                    if np.isfinite(a).any() else float("nan"),
                    "regen_median": float(np.nanmedian(b[np.isfinite(b)]))
                    if np.isfinite(b).any() else float("nan"),
                    "ks": st, "ks_p": pv, "n_kaggle": na, "n_regen": nb,
                })
        recs.sort(key=lambda d: (-(d["ks"] if d["ks"] == d["ks"] else -1)))
        per_label[lab] = recs
        print("\n--- %s   (Kaggle n=%d, regenerated n=%d)  10 most different "
              "features by KS ---" % (lab, int(oc.get(lab, 0)),
                                      int(rc.get(lab, 0))))
        print("%-22s %14s %14s %14s %14s %8s"
              % ("feature", "Kaggle mean", "regen mean", "Kaggle med",
                 "regen med", "KS"))
        for d in recs[:10]:
            print("%-22s %14.4g %14.4g %14.4g %14.4g %8s"
                  % (d["feature"][:22], d["kaggle_mean"], d["regen_mean"],
                     d["kaggle_median"], d["regen_median"],
                     "-" if d["ks"] != d["ks"] else "%.4f" % d["ks"]))
    out["per_feature_by_label"] = per_label

    if per_label:
        avg = {}
        for lab, recs in per_label.items():
            for d in recs:
                if d["ks"] == d["ks"]:
                    avg.setdefault(d["feature"], []).append(d["ks"])
        overall = sorted(((f, float(np.mean(v))) for f, v in avg.items()),
                         key=lambda kv: -kv[1])
        print("\n--- 10 most different features averaged over all shared "
              "labels ---")
        print("%-28s %10s" % ("feature", "mean KS"))
        for f, v in overall[:10]:
            print("%-28s %10.4f" % (f, v))
        out["overall_worst_features"] = [{"feature": f, "mean_ks": v}
                                         for f, v in overall]

    # ---- 3.5 row-level matching -----------------------------------------
    print("\n" + "=" * 78)
    print("3.5  ROW-LEVEL MATCHING")
    print("=" * 78)
    key_cols = [c for c in CFG.MATCH_KEY_FEATURES if c in common]
    dropped = [c for c in CFG.MATCH_KEY_FEATURES if c not in common]
    print("key features used (%d): %s" % (len(key_cols), ", ".join(key_cols)))
    if dropped:
        print("key features unavailable in both files, dropped: %s"
              % ", ".join(dropped))
    out["match_key_features"] = key_cols
    out["match_key_dropped"] = dropped

    matched_rows = pd.DataFrame(columns=["Source_Capture", "_row",
                                         "Label_Kaggle"])
    if not key_cols:
        print("no usable key features; row matching skipped")
        out["row_matching"] = {"skipped": "no usable key features"}
    else:
        ob = to_numeric_block(orig, key_cols)
        rb = to_numeric_block(regen, key_cols)
        o_ok = np.isfinite(ob.to_numpy(dtype="float64")).all(axis=1)
        r_ok = np.isfinite(rb.to_numpy(dtype="float64")).all(axis=1)
        print("original rows with a complete finite key    : %d / %d"
              % (int(o_ok.sum()), len(orig)))
        print("regenerated rows with a complete finite key : %d / %d"
              % (int(r_ok.sum()), len(regen)))

        def mkkeys(block, mask):
            """Integer key tuples. Plain Python tuples in a dict, not a
            pandas Index: an Index of tuples silently becomes a MultiIndex."""
            arr = np.rint(block.to_numpy(dtype="float64")[mask]).astype(
                "int64", copy=False)
            return list(map(tuple, arr))

        o_keys = mkkeys(ob, o_ok)
        r_keys = mkkeys(rb, r_ok)
        o_lab = orig.loc[o_ok, orig_label_col].astype(str).to_numpy()

        counts = Counter(o_keys)
        lut = {}
        for k, lab in zip(o_keys, o_lab):
            if counts[k] == 1:
                lut[k] = lab
        print("distinct keys in the original               : %d"
              % len(counts))
        print("keys unique in the original (usable)        : %d  (%.1f%% of "
              "its rows)" % (len(lut),
                             100.0 * len(lut) / max(1, len(o_keys))))

        r_idx = np.flatnonzero(r_ok)
        regen["Label_Kaggle"] = ""
        hit_idx, hit_lab = [], []
        for pos, k in zip(r_idx, r_keys):
            lab = lut.get(k)
            if lab is not None:
                hit_idx.append(pos)
                hit_lab.append(lab)
        if hit_idx:
            regen.loc[regen.index[hit_idx], "Label_Kaggle"] = hit_lab

        total_matched = len(hit_idx)
        print("\nregenerated rows matched to a unique original row: %d / %d "
              "(%.2f%%)" % (total_matched, len(regen),
                            100.0 * total_matched / max(1, len(regen))))

        print("\nmatch rate per regenerated label:")
        print("%-18s %12s %12s %10s" % ("my label", "rows", "matched", "%"))
        print("-" * 56)
        rows35 = []
        for lab in sorted(regen[CFG.LABEL_COLUMN].astype(str).unique()):
            sel = regen[CFG.LABEL_COLUMN].astype(str) == lab
            n = int(sel.sum())
            m = int((sel & (regen["Label_Kaggle"] != "")).sum())
            rows35.append({"label": lab, "rows": n, "matched": m,
                           "pct": 100.0 * m / n if n else 0.0})
            print("%-18s %12s %12s %9.2f%%"
                  % (lab, "{:,}".format(n), "{:,}".format(m),
                     100.0 * m / n if n else 0))

        print("\nmatch rate per capture:")
        print("%-10s %12s %12s %10s" % ("capture", "rows", "matched", "%"))
        print("-" * 48)
        rows35c = []
        for cap in [c[0] for c in CFG.CAPTURES]:
            sel = regen["Source_Capture"] == cap
            n = int(sel.sum())
            if not n:
                continue
            m = int((sel & (regen["Label_Kaggle"] != "")).sum())
            rows35c.append({"capture": cap, "rows": n, "matched": m,
                            "pct": 100.0 * m / n})
            print("%-10s %12s %12s %9.2f%%"
                  % (cap, "{:,}".format(n), "{:,}".format(m), 100.0 * m / n))

        print("\nconfusion table for matched rows  (rows = my label, "
              "columns = Kaggle label):")
        sub = regen[regen["Label_Kaggle"] != ""]
        if len(sub):
            ct = pd.crosstab(sub[CFG.LABEL_COLUMN].astype(str),
                             sub["Label_Kaggle"].astype(str))
            with pd.option_context("display.width", 200,
                                   "display.max_columns", 50):
                print(ct.to_string())
            agree = int(sum(ct.loc[l, l] for l in ct.index
                            if l in ct.columns))
            print("\nlabel agreement on matched rows: %d / %d (%.2f%%)"
                  % (agree, len(sub), 100.0 * agree / len(sub)))
            out["confusion_matched"] = ct.to_dict()
            out["label_agreement_matched"] = {
                "agree": agree, "total": int(len(sub)),
                "pct": 100.0 * agree / len(sub)}
        else:
            print("(no matched rows)")

        out["row_matching"] = {
            "total_regenerated": int(len(regen)),
            "total_matched": total_matched,
            "match_pct": 100.0 * total_matched / max(1, len(regen)),
            "original_rows_with_key": int(o_ok.sum()),
            "regen_rows_with_key": int(r_ok.sum()),
            "distinct_original_keys": int(len(counts)),
            "unique_original_keys": int(len(lut)),
            "per_label": rows35,
            "per_capture": rows35c,
        }
        matched_rows = regen.loc[regen["Label_Kaggle"] != "",
                                 ["Source_Capture", "_row", "Label_Kaggle"]]

    matched_rows.to_csv(CFG.metrics_path("03_matched_labels.csv"),
                        index=False)
    print("\nwrote %s (%d matched rows) for stage 04"
          % (CFG.metrics_path("03_matched_labels.csv"), len(matched_rows)))

    with open(CFG.metrics_path("03_comparison.json"), "w") as fh:
        json.dump(out, fh, indent=2, default=str)
    print("Metrics: %s" % CFG.metrics_path("03_comparison.json"))
    return 0

### 6.9 Stage 03 — run

Schema, rows per label, the microseconds units check, per-feature Kolmogorov–Smirnov
statistics per label, and row-level matching on the integer-feature key.

**How to read section 3.4:** if the four `Subflow` columns top the KS table, set
`CFG.SUBFLOW_MODE = "copy"` in section 2 and re-run stage 01. If the `Std`/`Var`
columns or `Down/Up Ratio` lead it, check `CFG.STD_DDOF` and
`CFG.JAVA_INTEGER_DIVISION`. If the `TotLen` / `Pkt Len` family leads it, try
`CFG.PKT_LEN_MODE = "frame"`.

**How to read section 3.5:** a low match rate is expected and is *not* evidence the
extraction is wrong — stage 02's packet reconciliation is the completeness check.
Section 4.6 of the generated report explains each cause.

In [ ]:
stage_03()

### 6.10 Stage 04 — consolidated CSVs  *(definition)*

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd


CONSOLIDATED = "MSCAD_consolidated.csv"
ML = "MSCAD_consolidated_ML.csv"


def stage_04():
    CFG.ensure_out_dirs()
    print("=" * 78)
    print("STAGE 04 -- consolidated CSVs")
    print("=" * 78)

    schema = build_schema()
    feats = schema.feature_display

    frames = []
    for key, _label, _desc in CFG.CAPTURES:
        p = CFG.raw_csv_path(key)
        if not os.path.isfile(p):
            print("  NOTE: %s missing, not included" % p)
            continue
        d = pd.read_csv(p, low_memory=False, skipinitialspace=True,
                        keep_default_na=False, na_values=["", "nan", "NaN"],
                        dtype={"Timestamp": str})
        d.columns = clean_names(d.columns)
        d["Source_Capture"] = key
        d["_row"] = np.arange(len(d))
        frames.append(d)
        print("  %-8s %10s flows" % (key, "{:,}".format(len(d))))
    if not frames:
        print("FATAL: no regenerated CSVs found in %s"
              % os.path.join(CFG.OUT_DIR, CFG.RAW_SUBDIR))
        return 2
    df = pd.concat(frames, ignore_index=True)
    print("  %-8s %10s flows total" % ("ALL", "{:,}".format(len(df))))

    # ---- Label_Kaggle from stage 03 -------------------------------------
    mpath = CFG.metrics_path("03_matched_labels.csv")
    df["Label_Kaggle"] = ""
    if os.path.isfile(mpath):
        m = pd.read_csv(mpath)
        if len(m):
            m["_row"] = m["_row"].astype(int)
            df = df.merge(m, on=["Source_Capture", "_row"], how="left",
                          suffixes=("", "_m"))
            df["Label_Kaggle"] = df["Label_Kaggle_m"].fillna("")
            df = df.drop(columns=[c for c in df.columns
                                  if c.endswith("_m")])
        print("  Label_Kaggle filled from %s: %d rows"
              % (os.path.basename(mpath), int((df["Label_Kaggle"] != "").sum())))
    else:
        print("  NOTE: %s not found; Label_Kaggle left empty everywhere "
              "(run 03_compare_with_kaggle.py first)" % mpath)

    # ---- sort and write the consolidated file ---------------------------
    df = df.sort_values("Timestamp", kind="mergesort").reset_index(drop=True)
    cols = (list(CFG.IDENTIFIER_COLUMNS) + list(feats)
            + [CFG.LABEL_COLUMN, "Label_Kaggle", "Source_Capture"])
    cols = [c for c in cols if c in df.columns]
    outp = os.path.join(CFG.OUT_DIR, CONSOLIDATED)
    df[cols].to_csv(outp, index=False)
    print("\nwrote %s" % outp)
    print("  shape : %d rows x %d columns" % (len(df), len(cols)))
    print("  labels:")
    lc = df[CFG.LABEL_COLUMN].astype(str).value_counts()
    for k, v in lc.items():
        print("      %-18s %10s" % (k, "{:,}".format(int(v))))
    print("  Label_Kaggle present : %s"
          % "{:,}".format(int((df["Label_Kaggle"].astype(str) != "").sum())))
    print("  per capture:")
    for k, v in df["Source_Capture"].value_counts().items():
        print("      %-10s %10s" % (k, "{:,}".format(int(v))))

    # ---- ML file --------------------------------------------------------
    print("\nbuilding the ML file ...")
    ml = df[[c for c in feats if c in df.columns]
            + [CFG.LABEL_COLUMN]].copy()
    for c in ml.columns[:-1]:
        if ml[c].dtype == object:
            ml[c] = pd.to_numeric(
                ml[c].replace({"Infinity": np.inf, "-Infinity": -np.inf,
                               "inf": np.inf, "-inf": -np.inf}),
                errors="coerce")
    n0 = len(ml)
    fcols = list(ml.columns[:-1])
    ml[fcols] = ml[fcols].replace([np.inf, -np.inf], np.nan)
    bad = ml[fcols].isna().any(axis=1)
    print("  rows with inf/NaN in a feature, dropped : %s"
          % "{:,}".format(int(bad.sum())))
    ml = ml[~bad]
    n1 = len(ml)
    ml = ml.drop_duplicates(subset=fcols)
    n2 = len(ml)
    print("  identical feature vectors dropped       : %s"
          % "{:,}".format(n1 - n2))
    mlp = os.path.join(CFG.OUT_DIR, ML)
    ml.to_csv(mlp, index=False)
    print("\nwrote %s" % mlp)
    print("  shape : %d rows x %d columns" % (ml.shape[0], ml.shape[1]))
    print("  labels:")
    mlc = ml[CFG.LABEL_COLUMN].astype(str).value_counts()
    for k, v in mlc.items():
        print("      %-18s %10s" % (k, "{:,}".format(int(v))))

    with open(CFG.metrics_path("04_consolidated.json"), "w") as fh:
        json.dump({
            "consolidated_path": outp,
            "consolidated_shape": [int(len(df)), int(len(cols))],
            "consolidated_columns": cols,
            "consolidated_labels": {str(k): int(v) for k, v in lc.items()},
            "consolidated_per_capture": {
                str(k): int(v) for k, v in
                df["Source_Capture"].value_counts().items()},
            "label_kaggle_filled": int(
                (df["Label_Kaggle"].astype(str) != "").sum()),
            "ml_path": mlp,
            "ml_shape": [int(ml.shape[0]), int(ml.shape[1])],
            "ml_labels": {str(k): int(v) for k, v in mlc.items()},
            "ml_rows_dropped_inf_nan": int(n0 - n1),
            "ml_rows_dropped_duplicate": int(n1 - n2),
        }, fh, indent=2, default=str)
    print("\nMetrics: %s" % CFG.metrics_path("04_consolidated.json"))
    return 0

### 6.11 Stage 04 — run

- `MSCAD_consolidated.csv` — identifiers + features + `Label` + `Label_Kaggle` +
  `Source_Capture`, sorted by `Timestamp`. **This is the file for PrefixSpan and the
  attack-state graph.**
- `MSCAD_consolidated_ML.csv` — features + `Label` only, infinities dropped and
  identical feature vectors de-duplicated, for RF / XGBoost / LSTM / CNN.

In [ ]:
stage_04()

### 6.12 Stage 05 — the report  *(definition)*

In [ ]:
import json
import os
import shutil
import sys
import datetime


HERE = os.getcwd()   # notebook folder: whatever .py/.ipynb is here gets copied
REPORT = "MSCAD_regeneration_report.md"


def load(name):
    p = CFG.metrics_path(name)
    if not os.path.isfile(p):
        return None
    with open(p) as fh:
        return json.load(fh)


def n(v):
    if v is None:
        return "not measured"
    if isinstance(v, bool):
        return "yes" if v else "no"
    if isinstance(v, int):
        return "{:,}".format(v)
    if isinstance(v, float):
        if v != v:
            return "NaN"
        return "%.4g" % v
    return str(v)


def table(headers, rows):
    out = ["| " + " | ".join(headers) + " |",
           "|" + "|".join(["---"] * len(headers)) + "|"]
    for r in rows:
        out.append("| " + " | ".join(str(c) for c in r) + " |")
    return "\n".join(out) + "\n"


def stage_05():
    CFG.ensure_out_dirs()
    s00, s01 = load("00_inputs.json"), load("01_extraction.json")
    s02, s03 = load("02_verification.json"), load("03_comparison.json")
    s04 = load("04_consolidated.json")

    L = []
    w = L.append
    cfgd = (s01 or {}).get("config", {})

    w("# MSCAD regeneration report\n")
    w("Regenerating flow-level features with flow identifiers from the raw "
      "MSCAD PCAPs, and measuring how close the result is to the original "
      "Kaggle `MSCAD.csv`.\n")
    w("- Generated: %s" % datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    w("- Output folder: `%s`" % CFG.OUT_DIR)
    w("- PCAP folder used: `%s`" % ((s01 or s00 or {}).get("pcap_dir")
                                    or "not recorded"))
    if (s01 or {}).get("pcap_dir_how"):
        w("  (located via: %s)" % s01["pcap_dir_how"])
    w("- Original Kaggle file: `%s`" % ((s03 or s00 or {}).get("original_file")
                                        or "not recorded"))
    w("")

    # ---------------- method ---------------------------------------------
    w("## 1. Method\n")
    w("### 1.1 Tool\n")
    w("Feature definitions come from the Python `cicflowmeter` package "
      "version 0.5.0, reimplemented in `mscad_flowmeter.py` as a streaming "
      "engine. The package's own field names are kept (`tot_fwd_pkts`, "
      "`flow_byts_s`, `cwr_flag_count`, ...) so the mapping to the MSCAD "
      "column names stays one-to-one and auditable. The numeric definitions "
      "follow the **Java** CICFlowMeter, because that is the tool that "
      "produced the Kaggle `MSCAD.csv` this report compares against.\n")
    w("A reimplementation rather than a monkey-patch was necessary: the "
      "package's feature classes take live scapy `Packet` objects and keep "
      "every packet of a flow in `Flow.packets`, so including non-TCP/UDP "
      "traffic, dropping the retained packets and reading real TCP flags all "
      "change its data model rather than its behaviour.\n")

    w("### 1.2 Patches applied\n")
    w(table(["#", "Problem in cicflowmeter 0.5.0", "What this pipeline does"],
            [["1", "CLI `-f file` crashes with `'bool' object has no "
                   "attribute 'split'`",
              "No CLI; the engine is a library called from Python."],
             ["2", "Every non-TCP/UDP packet is silently dropped "
                   "(`if \"TCP\" not in pkt and \"UDP\" not in pkt: return`), "
                   "so the ICMP flood would vanish",
              "All IPv4 packets are processed. ICMP and other IP protocols "
              "form flows keyed on (src, dst, 0, 0, proto): Src/Dst Port = 0, "
              "Protocol = 1 for ICMP. Non-first IP fragments carry no "
              "transport header and join the same port-0 flow, which is what "
              "keeps App-02's fragmented pings."],
             ["3", "BPF capture filter needs tcpdump, absent on Windows",
              "No capture filter is ever set."],
             ["4", "FIN is read as `\"F\" in pkt.flags`, which is the **IP** "
                   "flags field, not TCP",
              "FIN/SYN/RST/PSH/ACK/URG/ECE/CWR are read from the TCP flags "
              "byte (bits 0x01..0x80)."],
             ["5", "`Flow.__init__` stores the first packet and "
                   "`FlowSession.process` then calls `add_packet` on it "
                   "again, counting it twice",
              "Each packet is added exactly once."],
             ["6", "Every full scapy packet is retained in memory, which "
                   "kills the kernel on floods",
              "No packets and no per-packet records are retained. Every "
              "feature is an incremental accumulator (Welford mean/variance "
              "plus min/max/sum), so memory is O(open flows) and independent "
              "of flow length."],
             ["7", "`garbage_collect` rescans all open flows after every FIN: "
                   "quadratic on brute-force traffic",
              "Timeouts are checked on the packet's own flow, plus a periodic "
              "sweep every %s packets. Cost is O(packets), not "
              "O(packets x flows)." % n(CFG.SWEEP_EVERY_PACKETS)],
             ["8", "Timeouts do not match the Java tool",
              "Flow timeout %.0f s, idle timeout %.0f s, activity timeout "
              "%.0f s, clump/subflow timeout %.0f s."
              % (CFG.FLOW_TIMEOUT, CFG.IDLE_TIMEOUT, CFG.ACTIVITY_TIMEOUT,
                 CFG.CLUMP_TIMEOUT)],
             ["9", "**Found while reading the source.** "
                   "`Flow.update_subflow` compares `packet.time` with "
                   "`self.latest_timestamp` *after* that field has already "
                   "been set to `packet.time`, so the difference is always 0, "
                   "`update_active_idle` is never reached and every "
                   "Active/Idle feature is identically 0. `last_active` also "
                   "starts at 0, so a first idle value would have been ~1.7e9 "
                   "seconds.",
              "Active/Idle are computed properly, with both activity markers "
              "seeded from the flow's first packet."],
             ["10", "**Found while reading the source.** `get_statistics` "
                    "returns zeros unless `len(list) > 1`, so a two-packet "
                    "flow (one IAT) reports Flow IAT Mean/Max/Min = 0.",
              "Statistics are produced from one observation onwards."]]))

    w("### 1.3 Units and timestamps\n")
    w("Packet times are carried as **integer microseconds** end to end (a "
      "pcap record gives seconds + microseconds, so this is exact and free of "
      "floating-point drift). Every time-derived feature -- Flow Duration, "
      "all IAT features, all Active and Idle features -- is therefore emitted "
      "in **microseconds**, matching MSCAD.\n")
    w("Packet lengths are taken from the bytes in the capture; packets are "
      "never rebuilt.\n")
    w("MSCAD itself has no Timestamp column, so the `Timestamp` emitted here "
      "is this pipeline's own: the flow's first-packet time, "
      "`%s`%s, at microsecond precision, so the sequence-mining and "
      "attack-graph stages have a sortable, unambiguous ordering."
      % (CFG.TIMESTAMP_FORMAT, " in UTC" if CFG.TIMESTAMP_UTC else " in local time"))
    w("")

    w("### 1.4 Semantic choices\n")
    w("Three CICFlowMeter behaviours differ between the Java tool and the pip "
      "package. Each is a switch in `mscad_config.py`; the value used for "
      "this run is shown.\n")
    w(table(["Setting", "Value used", "Meaning"],
            [["`PKT_LEN_MODE`", "`%s`" % cfgd.get("pkt_len_mode",
                                                  CFG.PKT_LEN_MODE),
              "`payload` = L4 payload bytes (Java `getPayloadBytes()`); "
              "`frame` = whole captured frame (pip `len(packet)`). The CIC "
              "CSV family carries zeros in Fwd/Bwd Pkt Len Min, which only "
              "happens under payload semantics."],
             ["`HEADER_LEN_MODE`", "`%s`" % cfgd.get("header_len_mode",
                                                     CFG.HEADER_LEN_MODE),
              "`l4` = transport header only (Java: TCP `dataofs*4`, UDP 8); "
              "`ip` = IP header only (pip: `ihl*4`)."],
             ["`STD_DDOF`", "%s" % cfgd.get("std_ddof", CFG.STD_DDOF),
              "1 = bias-corrected (n-1), as Apache commons-math "
              "`SummaryStatistics` used by the Java tool; 0 = population (n), "
              "as numpy's default."],
             ["`SUBFLOW_MODE`", "`%s`" % cfgd.get("subflow_mode",
                                                  CFG.SUBFLOW_MODE),
              "`divide` = direction total / (1 + number of >1 s gaps), as "
              "Java; `copy` = Subflow == direction total, as pip. **This is "
              "the feature family whose Java semantics is least certain**; if "
              "the four Subflow columns dominate the section 3.4 KS table, "
              "switch to `copy` and re-run."],
             ["`JAVA_INTEGER_DIVISION`",
              n(cfgd.get("java_integer_division", CFG.JAVA_INTEGER_DIVISION)),
              "Java truncates long/long for Down/Up Ratio, the Subflow "
              "features and the bulk averages."],
             ["`INIT_WIN_DEFAULT`", "%s" % cfgd.get("init_win_default",
                                                    CFG.INIT_WIN_DEFAULT),
              "Init Fwd/Bwd Win Byts when that direction has no TCP packet. "
              "The CIC CSV family uses -1. These are the only features "
              "allowed to be negative."],
             ["`DROP_SINGLE_PACKET_FLOWS`",
              n(cfgd.get("drop_single_packet_flows",
                         CFG.DROP_SINGLE_PACKET_FLOWS)),
              "The Java tool discards a flow that expires on timeout with "
              "only one packet. Kept here, because Part 2 requires every IPv4 "
              "packet to be accounted for. Section 2.2 reports how many there "
              "are, which bounds this difference against Kaggle."],
             ["`ADD_FINAL_ACTIVE_SEGMENT`",
              n(cfgd.get("add_final_active_segment",
                         CFG.ADD_FINAL_ACTIVE_SEGMENT)),
              "Java does not add the final, unterminated Active segment when "
              "a flow is written, which is why many short flows have Active "
              "Mean = 0."]]))
    w("Zero-duration flows: the Java tool divides by 0.0 and so writes "
      "`Infinity` for the rate features (and `NaN` when the numerator is 0 "
      "too). This pipeline always writes `Infinity`, so that any `NaN` in the "
      "output is a genuine defect and the Part 2 NaN check stays meaningful. "
      "Section 2.2 reports the zero-duration flow count separately.\n")

    w("### 1.5 Labelling\n")
    w("`N-0` is labelled `Normal` throughout. In an attack capture, the "
      "attacker set is every Src IP whose total forward packet count reaches "
      "%.0f%% of the top sender's; a flow is given the capture's official "
      "label when its Src IP **or** Dst IP is an attacker, and `%s` "
      "otherwise.\n" % (100 * CFG.ATTACKER_SHARE, CFG.BENIGN_LABEL))

    # ---------------- schema ---------------------------------------------
    w("### 1.6 Output schema\n")
    if s01:
        w("The feature names and their order are read from the original "
          "Kaggle header, not hard-coded (`%s`).\n"
          % (s01.get("schema_source") or "unknown"))
        w("Feature columns: **%d**.\n" % len(s01.get("schema_features") or []))
        unmapped = s01.get("schema_unmapped") or []
        if unmapped:
            w("Original columns that could **not** be mapped to a "
              "CICFlowMeter feature (%d), and so are absent from the "
              "output:\n" % len(unmapped))
            for u in unmapped:
                w("- `%s`" % u)
        else:
            w("Every original column was mapped; none had to be dropped.\n")
        w("")
        w("Column order: " + ", ".join(
            "`%s`" % c for c in (list(CFG.IDENTIFIER_COLUMNS)
                                 + (s01.get("schema_features") or [])
                                 + [CFG.LABEL_COLUMN])) + "\n")
    else:
        w("Stage 01 metrics missing; schema not recorded.\n")

    # ---------------- inputs ---------------------------------------------
    w("## 2. Inputs and extraction\n")
    if s00 and s00.get("original_shape"):
        w("### 2.0 The original Kaggle file\n")
        w("- shape: **%d rows x %d columns**"
          % tuple(s00["original_shape"]))
        lc = s00.get("original_label_counts") or {}
        if lc:
            tot = sum(lc.values())
            w("")
            w(table(["Label", "rows", "% of file"],
                    [[k, n(v), "%.2f%%" % (100.0 * v / tot)]
                     for k, v in sorted(lc.items(), key=lambda kv: -kv[1])]
                    + [["**TOTAL**", n(tot), "100.00%"]]))
    if s01:
        w("### 2.1 Extraction per capture\n")
        rows = []
        for c in s01.get("captures", []):
            cc = c.get("counters", {})
            rows.append([c["capture"], c["official_label"],
                         n(cc.get("packets_read")), n(cc.get("ipv4_packets")),
                         n(cc.get("non_ipv4_packets")),
                         n(cc.get("ipv4_non_first_fragments")),
                         n(c.get("flows")),
                         "%.1f s" % c["seconds"] if c.get("seconds")
                         else "-"])
        w(table(["capture", "label", "packets read", "IPv4", "non-IPv4",
                 "frag (non-first)", "flows", "write pass"], rows))
        if s01.get("failures"):
            w("**Failures:**\n")
            for f in s01["failures"]:
                w("- `%s`: %s" % (f.get("capture"), f.get("error")))
            w("")
        w("#### Attacker detection\n")
        for c in s01.get("captures", []):
            if not c.get("top_senders"):
                continue
            w("**%s** (%s) -- attackers: %s\n"
              % (c["capture"], c["official_label"],
                 ", ".join("`%s`" % a for a in c.get("attackers", []))
                 or "none"))
            top = c["top_senders"][0]["fwd_pkts"] if c["top_senders"] else 0
            w(table(["Src IP", "total fwd packets", "% of top", "attacker?"],
                    [[t["src_ip"], n(t["fwd_pkts"]),
                      "%.2f%%" % (100.0 * t["fwd_pkts"] / top) if top else "-",
                      "yes" if t["src_ip"] in c.get("attackers", []) else ""]
                     for t in c["top_senders"][:5]]))
        w("#### Labels assigned per capture\n")
        rows = []
        for c in s01.get("captures", []):
            rows.append([c["capture"],
                         ", ".join("%s = %s" % (k, n(v)) for k, v in
                                   sorted((c.get("label_counts") or {}).items()))])
        w(table(["capture", "labels"], rows))

    # ---------------- part 2 ---------------------------------------------
    w("## 3. Verification of the extraction (Part 2)\n")
    if not s02:
        w("Stage 02 metrics missing; verification not run.\n")
    else:
        caps = s02.get("captures", [])
        w("### 3.1 Packet accounting\n")
        w("`IPv4 in flows` is the sum of Tot Fwd Pkts + Tot Bwd Pkts over "
          "every written flow. It must equal the independent counts exactly; "
          "the remainder up to `packets read` is ARP / IPv6 / non-IP, "
          "excluded by design.\n")
        rows = []
        for c in caps:
            ok = all(c.get(k) in (None, c.get("ipv4_in_flows"))
                     for k in ("tshark_ipv4", "second_ipv4", "ipv4_parsed"))
            rows.append([c["capture"], n(c.get("packets_read")),
                         n(c.get("ipv4_in_flows")), n(c.get("tshark_ipv4")),
                         n(c.get("second_ipv4")), n(c.get("non_ipv4")),
                         "**YES**" if ok else "**NO**"])
        w(table(["capture", "packets read", "IPv4 in flows", "tshark `-Y ip`",
                 "2nd-opinion count", "non-IPv4", "all match"], rows))
        notes = {c["capture"]: c.get("tshark_note") for c in caps
                 if c.get("tshark_ipv4") is None}
        for k, v in notes.items():
            w("- `%s`: tshark count unavailable -- %s" % (k, v))
        if notes:
            w("")

        w("### 3.2 Data sanity\n")
        rows = []
        for c in caps:
            rows.append([c["capture"], n(c.get("flows")),
                         n(c.get("duplicate_flows")), n(c.get("nan_cells")),
                         n(c.get("inf_cells")), n(c.get("negative_cells")),
                         n(c.get("minmax_violation_total")),
                         n(c.get("invalid_ports")),
                         n(c.get("zero_duration_flows")),
                         n(c.get("single_packet_flows")),
                         n(c.get("duplicate_feature_rows"))])
        w(table(["capture", "flows", "dup flows", "NaN", "inf", "negative",
                 "min>max", "bad ports", "zero-duration", "1-packet flows",
                 "identical feature rows"], rows))
        w("`dup flows` counts flows sharing a 5-tuple and start Timestamp "
          "(must be 0). `inf` cells are the zero-duration flows' Flow Byts/s, "
          "Flow Pkts/s, Fwd Pkts/s and Bwd Pkts/s -- normal CICFlowMeter "
          "behaviour, counted not removed. `identical feature rows` are "
          "expected in floods, scans and brute force; they are counted, never "
          "dropped. Init Fwd/Bwd Win Byts are excluded from the negative "
          "check because -1 is their defined 'no TCP packet' value.\n")
        probs = [(c["capture"], p) for c in caps for p in c.get("problems", [])]
        if probs:
            w("**Problems found (%d):**\n" % len(probs))
            for cap, p in probs:
                w("- `%s`: %s" % (cap, p))
            w("")
        else:
            w("**No problems found** across all captures: packet counts "
              "reconcile, no duplicate flows, no NaN, no negative values "
              "outside the two Init Win columns, no min>max inconsistency, "
              "all ports and protocol numbers valid.\n")

    # ---------------- part 3 ---------------------------------------------
    w("## 4. Comparison with the original Kaggle MSCAD.csv (Part 3)\n")
    if not s03:
        w("Stage 03 metrics missing; comparison not run.\n")
    else:
        sc = s03.get("schema", {})
        w("### 4.1 Schema\n")
        w("- original feature columns: **%d**"
          % len(sc.get("original_feature_columns") or []))
        w("- regenerated feature columns: **%d**"
          % len(sc.get("regenerated_feature_columns") or []))
        w("- same names and same order: **%s**"
          % n(sc.get("same_names_and_order")))
        for key, title in (("missing_in_regenerated",
                            "in the original but not regenerated"),
                           ("extra_in_regenerated",
                            "regenerated but not in the original"),
                           ("unmapped_original_columns",
                            "original columns not mappable to a "
                            "CICFlowMeter feature")):
            v = sc.get(key) or []
            if v:
                w("- %s (%d): %s" % (title, len(v),
                                     ", ".join("`%s`" % x for x in v)))
        w("")

        w("### 4.2 Rows per label\n")
        rows = [[r["label"], n(r["kaggle"]), n(r["regenerated"]),
                 "%+d" % r["difference"],
                 "-" if r["kaggle"] == 0 else "%+.1f%%" % r["diff_pct"]]
                for r in s03.get("rows_per_label", [])]
        w(table(["label", "Kaggle", "regenerated", "difference", "diff %"],
                rows))
        for k, t in (("labels_only_kaggle", "only in Kaggle"),
                     ("labels_only_regenerated", "only in the regenerated set")):
            v = s03.get(k) or []
            if v:
                w("- labels %s: %s" % (t, ", ".join("`%s`" % x for x in v)))
        w("")

        w("### 4.3 Units check\n")
        w("Both files should be in microseconds. A ratio near 10^6 or 10^-6 "
          "would mean one of them is in seconds.\n")
        rows = [[r["label"], r["feature"], n(r["kaggle_median"]),
                 n(r["regen_median"]), n(r["ratio"]), r["verdict"]]
                for r in s03.get("units_check", [])]
        w(table(["label", "feature", "Kaggle median", "regen median",
                 "ratio", "verdict"], rows))

        w("### 4.4 Per-feature distribution comparison\n")
        for lab, recs in (s03.get("per_feature_by_label") or {}).items():
            w("**%s** -- 10 most different features by the two-sample "
              "Kolmogorov-Smirnov statistic\n" % lab)
            w(table(["feature", "Kaggle mean", "regen mean", "Kaggle median",
                     "regen median", "KS", "n Kaggle", "n regen"],
                    [[d["feature"], n(d["kaggle_mean"]), n(d["regen_mean"]),
                      n(d["kaggle_median"]), n(d["regen_median"]),
                      n(d["ks"]), n(d["n_kaggle"]), n(d["n_regen"])]
                     for d in recs[:10]]))
        ow = s03.get("overall_worst_features") or []
        if ow:
            w("**Averaged over all shared labels** -- 10 most different "
              "features\n")
            w(table(["feature", "mean KS"],
                    [[d["feature"], n(d["mean_ks"])] for d in ow[:10]]))

        w("### 4.5 Row-level matching\n")
        rm = s03.get("row_matching") or {}
        if rm.get("skipped"):
            w("Skipped: %s\n" % rm["skipped"])
        else:
            w("Key: %s, each value rounded to an integer, using only keys "
              "that are unique in the original.\n"
              % ", ".join("`%s`" % c
                          for c in s03.get("match_key_features", [])))
            if s03.get("match_key_dropped"):
                w("Key features unavailable in both files and therefore "
                  "dropped: %s\n"
                  % ", ".join("`%s`" % c for c in s03["match_key_dropped"]))
            w("- original rows with a complete finite key: %s"
              % n(rm.get("original_rows_with_key")))
            w("- distinct keys in the original: %s"
              % n(rm.get("distinct_original_keys")))
            w("- keys unique in the original (usable): %s"
              % n(rm.get("unique_original_keys")))
            w("- regenerated rows matched: **%s / %s (%.2f%%)**"
              % (n(rm.get("total_matched")), n(rm.get("total_regenerated")),
                 rm.get("match_pct", 0.0)))
            w("")
            w("Match rate per regenerated label\n")
            w(table(["my label", "rows", "matched", "%"],
                    [[r["label"], n(r["rows"]), n(r["matched"]),
                      "%.2f%%" % r["pct"]] for r in rm.get("per_label", [])]))
            w("Match rate per capture\n")
            w(table(["capture", "rows", "matched", "%"],
                    [[r["capture"], n(r["rows"]), n(r["matched"]),
                      "%.2f%%" % r["pct"]]
                     for r in rm.get("per_capture", [])]))
            conf = s03.get("confusion_matched")
            if conf:
                kcols = sorted(conf.keys())
                mine = sorted({r for c in conf.values() for r in c.keys()})
                w("Confusion table for matched rows (rows = my label, "
                  "columns = Kaggle label)\n")
                w(table(["my label \\ Kaggle"] + kcols,
                        [[m] + [n(conf[k].get(m, 0)) for k in kcols]
                         for m in mine]))
                ag = s03.get("label_agreement_matched") or {}
                if ag:
                    w("Label agreement on matched rows: **%s / %s "
                      "(%.2f%%)**\n" % (n(ag.get("agree")), n(ag.get("total")),
                                        ag.get("pct", 0.0)))

        w("### 4.6 Why the two files differ\n")
        w("**Measured in this run** (the numbers are in the tables above):\n")
        w("- Row counts per label, and the total row-count difference "
          "(section 4.2).")
        w("- Which features' distributions diverge most, per label "
          "(section 4.4).")
        w("- What fraction of regenerated flows matches a unique original row "
          "on the integer key, and how the labels agree on those matches "
          "(section 4.5).")
        w("- How many regenerated flows carry only one packet, and how many "
          "have zero duration (section 3.2).")
        w("")
        w("**Explanations, strongest first.** The first three are mechanical "
          "and their size is bounded by numbers in this report; the rest are "
          "reasoned but not individually measured here.\n")
        w("1. **Single-packet flows (measured, mechanical).** The Java tool "
          "discards a flow that expires on timeout with only one packet; this "
          "pipeline keeps it, because Part 2 requires every IPv4 packet to be "
          "accounted for. In a port scan that is a large, one-sided "
          "difference: see the 1-packet flow column in section 3.2. Set "
          "`DROP_SINGLE_PACKET_FLOWS = True` to drop them and re-run if you "
          "want the closer row count instead of the packet-conservation "
          "guarantee.")
        w("2. **ICMP (mechanical, by design).** MSCAD was built with a tool "
          "that processes only TCP and UDP, so the App-02 ICMP flood "
          "contributes few or no rows to the Kaggle file, while it does "
          "contribute here. Compare the ICMP_Flood row counts in section 4.2 "
          "and the protocol mix in stage 02's output.")
        w("3. **The Benign rule (mechanical, ours).** Our `Benign` label is a "
          "rule we chose -- a flow in an attack capture involving no attacker "
          "IP -- not a label the dataset publishes. Any disagreement between "
          "`Benign` and a Kaggle attack label in the section 4.5 confusion "
          "table comes from this rule, not from the features.")
        w("4. **Flow cutting.** Identical timeouts still do not guarantee "
          "identical flow boundaries: whether a FIN (or RST) closes a flow, "
          "whether the 120 s cut is measured from the flow's start or its last "
          "packet, and the order in which expiry is checked, all move packets "
          "between flows. Every feature of the affected flows changes with "
          "them, so a flow-boundary difference shows up as a distribution "
          "difference rather than as a missing row.")
        w("5. **Statistical conventions.** Java's `SummaryStatistics` uses "
          "the bias-corrected (n-1) variance and truncates long/long "
          "division; numpy does neither. This run used `STD_DDOF = %s` and "
          "`JAVA_INTEGER_DIVISION = %s`. If the Std/Var columns or Down/Up "
          "Ratio lead the section 4.4 table, this is the first thing to "
          "check." % (cfgd.get("std_ddof", CFG.STD_DDOF),
                      n(cfgd.get("java_integer_division",
                                 CFG.JAVA_INTEGER_DIVISION))))
        w("6. **Packet-length basis.** `PKT_LEN_MODE = %r` was used. Under "
          "`payload` the length family counts L4 payload bytes, as the Java "
          "tool does; under `frame` it counts whole captured frames, as the "
          "pip package does. The two differ by roughly the header size on "
          "every packet, so picking the wrong one shifts eight features at "
          "once (TotLen, Pkt Len Min/Max/Mean/Std/Var, Pkt Size Avg, Seg Size "
          "Avg)." % cfgd.get("pkt_len_mode", CFG.PKT_LEN_MODE))
        w("7. **Subflow semantics (acknowledged uncertainty).** See section "
          "1.4. If the four Subflow columns top the section 4.4 table, switch "
          "`SUBFLOW_MODE` and re-run.")
        w("8. **What the original did that we cannot see.** The published "
          "MSCAD.csv may have been filtered, de-duplicated, re-ordered or "
          "partly hand-labelled before release, and it carries no "
          "identifiers, so a row that fails to match cannot be traced back to "
          "a capture. This is a limit of the comparison, not a measurement.")
        w("")
        w("A low row-level match rate is therefore expected and is **not** "
          "evidence that the extraction is wrong: section 3.1's packet "
          "reconciliation is the check that the extraction is complete, and "
          "section 4.4's distributions are the check that it is faithful.\n")

    # ---------------- part 4 ---------------------------------------------
    w("## 5. Final files (Part 4)\n")
    if not s04:
        w("Stage 04 metrics missing; consolidated files not built.\n")
    else:
        w("### `MSCAD_consolidated.csv`\n")
        w("- shape: **%d rows x %d columns**" % tuple(s04["consolidated_shape"]))
        w("- columns: `Src IP`, `Dst IP`, `Src Port`, `Dst Port`, "
          "`Protocol`, `Timestamp`, the MSCAD features, `Label`, "
          "`Label_Kaggle`, `Source_Capture`")
        w("- sorted by `Timestamp`")
        w("- `Label_Kaggle` filled on **%s** rows (the official Kaggle label "
          "where section 4.5 matched the row; empty elsewhere)"
          % n(s04.get("label_kaggle_filled")))
        w("")
        w(table(["Label", "rows"],
                [[k, n(v)] for k, v in
                 sorted((s04.get("consolidated_labels") or {}).items(),
                        key=lambda kv: -kv[1])]))
        w(table(["Source_Capture", "rows"],
                [[k, n(v)] for k, v in
                 (s04.get("consolidated_per_capture") or {}).items()]))
        w("### `MSCAD_consolidated_ML.csv`\n")
        w("For classifier training: the features + `Label` only.\n")
        w("- shape: **%d rows x %d columns**" % tuple(s04["ml_shape"]))
        w("- rows dropped for an infinite or missing feature: **%s**"
          % n(s04.get("ml_rows_dropped_inf_nan")))
        w("- rows dropped as an identical feature vector: **%s**"
          % n(s04.get("ml_rows_dropped_duplicate")))
        w("")
        w(table(["Label", "rows"],
                [[k, n(v)] for k, v in
                 sorted((s04.get("ml_labels") or {}).items(),
                        key=lambda kv: -kv[1])]))
        w("Two caveats on this file. Dropping rows with an infinite feature "
          "removes **every zero-duration flow**, which in a port scan is most "
          "of the scan; and de-duplicating identical feature vectors removes "
          "real flood, scan and brute-force traffic, because those attacks "
          "genuinely produce repeated flows. Both were asked for, and both "
          "shift the class balance -- compare the label counts above against "
          "the consolidated file's. Use `MSCAD_consolidated.csv`, which keeps "
          "every flow and its identifiers, for the sequence-mining and "
          "attack-graph stages, and this file only for the flow-level "
          "classifiers. If the class balance matters more than the "
          "de-duplication, cap the rate features instead of dropping their "
          "rows.\n")

    # ---------------- thesis paragraph -----------------------------------
    w("## 6. Paragraph for the thesis methodology\n")
    caps_n = len((s01 or {}).get("captures", []))
    total_flows = sum(c.get("flows", 0) for c in (s01 or {}).get("captures", []))
    pkt_list = [c.get("counters", {}).get("packets_read", 0)
                for c in (s01 or {}).get("captures", [])]
    total_pkts = sum(pkt_list)
    biggest_pkts = max(pkt_list) if pkt_list else 0
    w("> The MSCAD distribution publishes flow-level features without flow "
      "identifiers, which makes it unusable for sequential pattern mining and "
      "for attack-state graph construction: neither can be built without "
      "knowing which host pair a flow belongs to or when it occurred. The %d "
      "raw captures were therefore re-processed into flow records carrying "
      "both the identifiers and the published feature set. Feature "
      "definitions were taken from CICFlowMeter and implemented as a "
      "streaming engine with incremental accumulators, so memory "
      "is independent of flow length and the largest capture (%s packets) "
      "was processed without buffering packets. Flows were cut at a 120 s flow "
      "timeout and a 240 s idle timeout, with a 5 s activity threshold, and "
      "every time-derived feature was emitted in microseconds to match the "
      "published file. Unlike the reference implementations, all IPv4 "
      "protocols were retained, including ICMP and non-first IP fragments, so "
      "that the volumetric ICMP flood is represented rather than discarded. "
      "Extraction completeness was verified by reconciling the per-flow "
      "packet counts against an independent packet count of each capture, and "
      "fidelity was assessed against the published file by comparing "
      "per-label feature distributions with the two-sample "
      "Kolmogorov-Smirnov statistic and by matching individual flows on a key "
      "of integer-valued features. The result is %s labelled flow records "
      "with source and destination addresses, ports, protocol and timestamp, "
      "which is the input to the sequential and graph-based stages of the "
      "framework."
      % (caps_n or 6, n(biggest_pkts) if biggest_pkts else "largest",
         n(total_flows) if total_flows else "the full set of"))
    w("")

    w("## 7. Reproducing this\n")
    w("The scripts are copied into `%s` alongside this report. From a `cmd` "
      "prompt:\n" % os.path.join(CFG.OUT_DIR, "scripts"))
    w("```")
    w("cd /d %s" % os.path.join(CFG.OUT_DIR, "scripts"))
    w("set PY=C:\\Users\\banot\\AppData\\Local\\Programs\\Python\\Python313"
      "\\python.exe")
    w("%PY% 00_inspect_inputs.py")
    w("%PY% 01_pcap_to_csv.py N-0")
    w("%PY% 01_pcap_to_csv.py")
    w("%PY% 02_verify.py")
    w("%PY% 03_compare_with_kaggle.py")
    w("%PY% 04_consolidate.py")
    w("%PY% 05_report.py")
    w("```")
    w("")
    w("Or simply `run_all.bat`. All paths and every semantic switch live in "
      "`mscad_config.py`.\n")

    path = os.path.join(CFG.OUT_DIR, REPORT)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print("wrote %s (%d lines)" % (path, len(L)))

    # ---- copy the scripts ----------------------------------------------
    dest = os.path.join(CFG.OUT_DIR, "scripts")
    os.makedirs(dest, exist_ok=True)
    copied = []
    for f in sorted(os.listdir(HERE)):
        if f.endswith((".py", ".bat", ".md")) and f != REPORT:
            shutil.copy2(os.path.join(HERE, f), os.path.join(dest, f))
            copied.append(f)
    print("copied %d script(s) to %s" % (len(copied), dest))
    for f in copied:
        print("    %s" % f)
    return 0

### 6.13 Stage 05 — run

Writes `MSCAD_regeneration_report.md` into the output folder: the method, every table
above, the file descriptions, and a paragraph you can adapt for your thesis
methodology. It also copies whatever scripts/notebooks are in the current folder into
`<out>\scripts` so the run is reproducible.

In [ ]:
stage_05()

## 7. Load the result and sanity-check it here

In [ ]:
cons = os.path.join(CFG.OUT_DIR, "MSCAD_consolidated.csv")
mlf = os.path.join(CFG.OUT_DIR, "MSCAD_consolidated_ML.csv")
full = None

if os.path.isfile(cons):
    full = pd.read_csv(cons, low_memory=False)
    print("MSCAD_consolidated.csv :", full.shape)
    print("\nlabels:")
    print(full["Label"].value_counts().to_string())
    print("\nper capture:")
    print(full["Source_Capture"].value_counts().to_string())
    print("\nLabel_Kaggle filled on %d rows"
          % int(full["Label_Kaggle"].notna().sum()))
    print("\ntime span: %s  ->  %s"
          % (full["Timestamp"].min(), full["Timestamp"].max()))
    print("\nfirst rows:")
    print(full[["Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol",
                "Timestamp", "Label", "Source_Capture"]].head(8).to_string())
else:
    print("not found yet:", cons)

if os.path.isfile(mlf):
    ml = pd.read_csv(mlf, low_memory=False)
    print("\nMSCAD_consolidated_ML.csv :", ml.shape)
    print(ml["Label"].value_counts().to_string())
    print("\nNOTE: dropping infinite rows removes every zero-duration flow,")
    print("which in a port scan is most of the scan; and de-duplicating identical")
    print("feature vectors removes real flood/scan/brute-force traffic. Compare the")
    print("two label tables above before training -- the class balance differs.")

## 8. The sequence view — what the regeneration was for

Grouping the consolidated flows per host pair, in time order, gives the labelled
sequences PrefixSpan takes as input. None of this is possible from the Kaggle file,
which has no identifiers and no timestamp.

In [ ]:
if full is not None:
    seq = (full.sort_values("Timestamp")
               .groupby(["Src IP", "Dst IP"])["Label"]
               .apply(list))
    print("host pairs with at least one flow    :", len(seq))
    multi = seq[seq.map(len) > 1]
    print("host pairs with a multi-flow sequence:", len(multi))
    print("\nlongest 10 sequences (first 12 flows shown):")
    for (s, d), labs in multi.sort_values(key=lambda x: x.map(len),
                                          ascending=False).head(10).items():
        print("  %-16s -> %-16s  n=%-7d %s%s"
              % (s, d, len(labs), " > ".join(labs[:12]),
                 " > ..." if len(labs) > 12 else ""))

    print("\ndistinct attack-stage transitions observed per host pair:")
    from collections import Counter as _C
    trans = _C()
    for labs in multi:
        for a, b in zip(labs, labs[1:]):
            if a != b:
                trans[(a, b)] += 1
    if trans:
        for (a, b), n in trans.most_common(20):
            print("  %-16s -> %-16s  %d" % (a, b, n))
    else:
        print("  (none: no host pair changed label between consecutive flows)")
else:
    print("run stage 04 first")